In [ ]:
%matplotlib inline
import numpy as np
import h5py
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import gaussian_kde
from matplotlib.lines import Line2D

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['svg.fonttype']='none'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Arial'
plt.rcParams['mathtext.it'] = 'Arial:italic'
plt.rcParams['mathtext.bf'] = 'Arial:bold'

subgraph_dir = Path('fig/Subgraph')
subgraph_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
cmip5_tas = np.zeros((18, 106))
CMIP5_address = 'data/CMIP5_annual_tas.h5py'
f = h5py.File(CMIP5_address,'r')
cmip5_model_name_dirs = list(f.keys())
for i in range (len(cmip5_model_name_dirs)):
    cmip5_tas[i,:] = f[cmip5_model_name_dirs[i]][:]
f.close()

cmip6_tas = np.zeros((23, 115))
CMIP6_address = 'data/CMIP6_annual_tas.h5py'
f = h5py.File(CMIP6_address,'r')
cmip6_model_name_dirs = list(f.keys())
for i in range (len(cmip6_model_name_dirs)):
    cmip6_tas[i,:] = f[cmip6_model_name_dirs[i]][:]
f.close()

HadCRUT5_address = 'data/HadCRUT5_annual_tas.h5py'
f = h5py.File(HadCRUT5_address,'r')
HadCRUT5_1900_2014 = f['HadCRUT5_1900-2014'][:]
HadCRUT5_1900_2019 = f['HadCRUT5_1900-2019'][:]
f.close()

In [ ]:
def rolling_psi(tas, window_size):
    windows = np.lib.stride_tricks.sliding_window_view(tas, window_size)
    years = np.arange(window_size)
    slope, intercept = np.polyfit(years, windows.T, 1)
    detrended = windows - (slope[:, None] * years + intercept[:, None])
    left, right = detrended[:, :-1], detrended[:, 1:]
    left = left - left.mean(axis=1, keepdims=True)
    right = right - right.mean(axis=1, keepdims=True)
    numerator = np.sum(left * right, axis=1)
    scale = np.linalg.norm(left, axis=1) * np.linalg.norm(right, axis=1)
    alpha = np.divide(numerator, scale, out=np.full(len(windows), np.nan), where=scale > 0)
    valid = (alpha > 0) & (alpha < 1)
    psi = np.full(len(windows), np.nan)
    psi[valid] = detrended[valid].std(axis=1) / np.sqrt(-np.log(alpha[valid]))
    return psi

def record_mean_psi(windows):
    if not np.all(np.any(np.isfinite(windows), axis=-1)):
        raise ValueError("A temperature record has no valid Ψ windows.")
    return np.nanmean(windows, axis=-1)

HadCRUT5_psi = rolling_psi(HadCRUT5_1900_2014, 55)
assert len(HadCRUT5_psi) == 61 and np.isfinite(HadCRUT5_psi).all()

In [ ]:
model_ecs = pd.read_csv('data/models.csv').set_index('model', verify_integrity=True)
cmip5_ecs = model_ecs.loc[cmip5_model_name_dirs, 'ecs'].to_numpy()
cmip5_psi_windows = np.stack([rolling_psi(tas, 55) for tas in cmip5_tas])
assert cmip5_psi_windows.shape == (18, 52) and np.isfinite(cmip5_psi_windows).all()
cmip5_psi = record_mean_psi(cmip5_psi_windows)

In [ ]:
cmip6_ecs = model_ecs.loc[cmip6_model_name_dirs, 'ecs'].to_numpy()
cmip6_psi_windows = np.stack([rolling_psi(tas, 55) for tas in cmip6_tas])
assert cmip6_psi_windows.shape == (23, 61) and np.isfinite(cmip6_psi_windows).all()
cmip6_psi = record_mean_psi(cmip6_psi_windows)

In [ ]:
def fit_line(x_fit, y_fit,color,level):
    n = len(x_fit)
    slope, intercept = np.polyfit(x_fit, y_fit, deg=1)
    xs = np.linspace(float(np.min(x_fit)), float(np.max(x_fit)), 120)
    yhat = slope * xs + intercept
    residuals = y_fit - (slope * x_fit + intercept)
    sxx = float(np.sum((x_fit - float(np.mean(x_fit))) ** 2))
    if sxx > 0 and n > 2:
        sigma = float(np.sqrt(np.sum(residuals**2) / (n - 2)))
        pred_se = sigma * np.sqrt(1.0 + 1.0 / n + ((xs - float(np.mean(x_fit))) ** 2) / sxx)
        crit = float(stats.t.ppf((1.0 + level) / 2.0, df=n-2))
        plt.fill_between(xs, yhat - crit * pred_se, yhat + crit * pred_se, color=color, alpha=0.11, linewidth=0, zorder=1)
    plt.plot(xs, yhat, color=color, linewidth=1.15, zorder=2)

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axvline(x=np.mean(HadCRUT5_psi), color='#000000', linestyle='--',)
plt.axvspan(np.mean(HadCRUT5_psi)-np.std(HadCRUT5_psi), np.mean(HadCRUT5_psi)+np.std(HadCRUT5_psi), color="#E9E4E4", alpha=0.5)

plt.scatter(cmip6_psi, cmip6_ecs,facecolors="#BB6644", edgecolors='#000000',s=100,linewidths=1.2)
fit_line(cmip6_psi, cmip6_ecs,color="#BB6644",level=0.95)
r, p = stats.pearsonr(cmip6_psi, cmip6_ecs)
plt.text(0.95,0.19,f'CMIP6',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.11,f'r = {r:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.03,f'p = {p:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)

plt.xlim(0.04,0.4)
plt.ylim(1.5,6.1)
plt.yticks(np.arange(2,6.1,1))
plt.xlabel(r'$\Psi$ (K)', fontsize=18)
plt.ylabel('ECS (K)', fontsize=18)  
plt.tick_params(axis='both', which='major', labelsize=18)
plt.savefig(subgraph_dir / 'FIG 1b.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axvline(x=np.mean(HadCRUT5_psi), color='#000000', linestyle='--',)
plt.axvspan(np.mean(HadCRUT5_psi)-np.std(HadCRUT5_psi), np.mean(HadCRUT5_psi)+np.std(HadCRUT5_psi), color="#E9E4E4", alpha=0.5)

plt.scatter(cmip5_psi, cmip5_ecs,facecolors="#03727E", edgecolors='#000000',s=100,linewidths=1.2)
fit_line(cmip5_psi, cmip5_ecs,color="#03727E",level=0.95)
r, p = stats.pearsonr(cmip5_psi, cmip5_ecs)
plt.text(0.95,0.18,f'CMIP5',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.10,f'r = {r:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.01,f'p = {p:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)

plt.xlim(0.04,0.4)
plt.ylim(1.5,6.1)
plt.yticks(np.arange(2,6.1,1))
plt.xlabel(r'$\Psi$ (K)', fontsize=18)
plt.ylabel('ECS (K)', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG 1a.svg', transparent=True,bbox_inches='tight')

In [ ]:
forcing_address = 'data/AR6_ERF_1750-2019.csv'
forcing_df = pd.read_csv(forcing_address).set_index('year', verify_integrity=True)
forcing_columns = ['co2', 'aerosol', 'total_natural']

def forcing_1900_to(end_year):
    values = forcing_df.loc[np.arange(1900, end_year + 1), forcing_columns].to_numpy()
    if not np.isfinite(values).all():
        raise ValueError("The forcing records contain nonfinite values.")
    return values

forcing_1900_2005 = forcing_1900_to(2005)
forcing_1900_2014 = forcing_1900_to(2014)

In [ ]:
def _arx_design(values, forcing_matrix, order, target_start=None):
    start = order if target_start is None else target_start
    rows = []
    for i in range(start, values.size):
        lags = [values[i - lag] for lag in range(1, order + 1)]
        rows.append([1.0, *lags, *forcing_matrix[i].tolist()])
    return np.array(rows, dtype=float)

def _spectral_radius(ar_coef):
    order = ar_coef.size
    companion = np.zeros((order, order))
    companion[0, :] = ar_coef
    if order > 1:
        companion[1:, :-1] = np.eye(order - 1)
    return float(np.max(np.abs(np.linalg.eigvals(companion))))

def fit_arx(values, forcing_matrix, order, target_start=None):
    start = order if target_start is None else target_start
    design = _arx_design(values, forcing_matrix, order, target_start=start)
    target = values[start:]
    coef, *_ = np.linalg.lstsq(design, target, rcond=None)
    resid = target - design @ coef
    n, k = resid.size, coef.size
    rss = float(np.sum(resid**2))
    sigma2 = max(rss / n, np.finfo(float).tiny)
    aicc = n * np.log(sigma2) + 2*k + (2*k*(k+1)) / max(n - k - 1, 1e-9)
    ar_coef = coef[1:1+order]
    sr = _spectral_radius(ar_coef)
    return {
        "order": order,
        "intercept": float(coef[0]),
        "ar_coef": ar_coef,
        "forcing_coef": coef[1+order:],   
        "spectral_radius": sr,
        "is_stationary": sr < 1.0,
        "aicc": aicc,
    }

def step_response_curve(ar_coef, beta_co2, horizon=150):
    order = ar_coef.size
    state = np.zeros(order)
    curve = np.zeros(horizon)
    for i in range(horizon):
        next_val = float(ar_coef @ state) + beta_co2
        state = np.concatenate([[next_val], state[:-1]])
        curve[i] = state[0]
    return curve

In [ ]:
F2XCO2 = 3.93

def select_arx(tas, forcing, target_start=None):
    fits = [fit_arx(tas, forcing, order=k, target_start=target_start) for k in range(1, 4)]
    stationary = [fit for fit in fits if fit["is_stationary"]]
    if not stationary:
        raise ValueError("No stationary ARX model found.")
    return min(stationary, key=lambda fit: fit["aicc"])

def Rco2_step(tas, forcing, horizon=150, target_start=None):
    fit = select_arx(tas, forcing, target_start=target_start)
    return F2XCO2 * step_response_curve(fit["ar_coef"], fit["forcing_coef"][0], horizon=horizon)

def Rco2_main(tas, forcing, horizon):
    return Rco2_step(tas, forcing, horizon=horizon)[horizon - 1]

In [ ]:
cmip5_Rco2_step = np.stack([Rco2_step(tas, forcing_1900_2005) for tas in cmip5_tas])
cmip6_Rco2_step = np.stack([Rco2_step(tas, forcing_1900_2014) for tas in cmip6_tas])
HadCRUT5_Rco2_step = Rco2_step(HadCRUT5_1900_2014, forcing_1900_2014)

cmip5_Rco2 = cmip5_Rco2_step[:, 54]
cmip6_Rco2 = cmip6_Rco2_step[:, 54]
HadCRUT5_Rco2 = HadCRUT5_Rco2_step[54]

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axvline(x=np.mean(HadCRUT5_Rco2), color='#000000', linestyle='--',)
plt.scatter(cmip6_Rco2, cmip6_ecs,facecolors="#BB6644", edgecolors='#000000',s=100,linewidths=1.2)
fit_line(cmip6_Rco2, cmip6_ecs,color="#BB6644",level=0.95)
r, p = stats.pearsonr(cmip6_Rco2, cmip6_ecs)
plt.text(0.95,0.19,f'CMIP6',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.11,f'r = {r:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.03,'p < 0.001' if p < 0.001 else f'p = {p:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)

plt.ylim(1.5,6.1)
plt.xlabel(r'$R$ (K)', fontsize=18)
plt.ylabel('ECS (K)', fontsize=18)
plt.yticks(np.arange(2,6.1,1))
plt.xticks(np.arange(2,5.1,1))
plt.tick_params(axis='both', which='major', labelsize=18)
plt.savefig(subgraph_dir / 'FIG 1d.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axvline(x=np.mean(HadCRUT5_Rco2), color='#000000', linestyle='--',)
plt.scatter(cmip5_Rco2, cmip5_ecs,facecolors="#03727E", edgecolors='#000000',s=100,linewidths=1.2)
fit_line(cmip5_Rco2, cmip5_ecs,color="#03727E",level=0.95)
r, p = stats.pearsonr(cmip5_Rco2, cmip5_ecs)
plt.text(0.3,0.9,f'CMIP5',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.3,0.82,f'r = {r:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.3,0.74,f'p = {p:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)

plt.ylim(1.5,6.1)
plt.xlabel(r'$R$ (K)', fontsize=18)
plt.ylabel('ECS (K)', fontsize=18)
plt.yticks(np.arange(2,6.1,1))
plt.xticks(np.arange(2,5.1,1))
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG 1c.svg', transparent=True,bbox_inches='tight')

In [ ]:
def residual_pair(psi, ecs, response):
    E0 = np.polyfit(psi, ecs, 1)
    R0 = np.polyfit(psi, response, 1)
    delta_E = ecs - np.polyval(E0, psi)
    delta_R = response - np.polyval(R0, psi)
    return E0, R0, delta_E, delta_R

def fit_line_confidence(x_fit, y_fit,color,level):
    n = len(x_fit)
    sxx = float(np.sum(x_fit**2))
    slope = float(np.sum(x_fit * y_fit) / sxx)
    xs = np.linspace(float(np.min(x_fit)), float(np.max(x_fit)), 120)
    yhat = slope * xs
    residuals = y_fit - slope * x_fit
    sigma = float(np.sqrt(np.sum(residuals**2) / (n - 3)))
    slope_se = sigma / np.sqrt(sxx)
    crit = float(stats.t.ppf((1.0 + level) / 2.0, df=n - 3))
    margin = crit * slope_se * np.abs(xs)
    plt.fill_between(xs, yhat - margin, yhat + margin, color=color, alpha=0.11, linewidth=0, zorder=1)
    plt.plot(xs, yhat, color=color, linewidth=1.5, zorder=2)

In [ ]:
cmip5_E0, cmip5_R0, cmip5_delta_E, cmip5_delta_R = residual_pair(cmip5_psi, cmip5_ecs, cmip5_Rco2)
fit_E_psi_cmip5_slope, fit_E_psi_cmip5_intercept = cmip5_E0
fit_R_psi_cmip5_slope, fit_R_psi_cmip5_intercept = cmip5_R0
cmip5_cg = np.corrcoef(cmip5_delta_E, cmip5_delta_R)[0, 1]
cmip5_gamma = float(np.sum(cmip5_delta_R * cmip5_delta_E) / np.sum(cmip5_delta_R**2))

In [ ]:
cmip6_E0, cmip6_R0, cmip6_delta_E, cmip6_delta_R = residual_pair(cmip6_psi, cmip6_ecs, cmip6_Rco2)
fit_E_psi_cmip6_slope, fit_E_psi_cmip6_intercept = cmip6_E0
fit_R_psi_cmip6_slope, fit_R_psi_cmip6_intercept = cmip6_R0
cmip6_cg = np.corrcoef(cmip6_delta_E, cmip6_delta_R)[0, 1]
cmip6_gamma = float(np.sum(cmip6_delta_R * cmip6_delta_E) / np.sum(cmip6_delta_R**2))

In [ ]:
PERMUTATIONS = 50_000
PERMUTATION_SEED = 20261111

def permutation_test(delta_E, delta_R, group, window_size, case=0):
    rng = np.random.default_rng([PERMUTATION_SEED, group, window_size, case])
    e = delta_E - delta_E.mean()
    r = delta_R - delta_R.mean()
    norm = np.linalg.norm(e) * np.linalg.norm(r)
    shuffled_r = rng.permuted(np.broadcast_to(r, (PERMUTATIONS, len(r))), axis=1)
    null = (shuffled_r @ e) / norm
    observed = abs(r @ e / norm)
    p_perm = (1 + np.count_nonzero(np.abs(null) >= observed)) / (PERMUTATIONS + 1)
    return p_perm, null

In [ ]:
cmip5_p_perm, cmip5_null_model = permutation_test(cmip5_delta_E, cmip5_delta_R, 5, 55)
cmip6_p_perm, cmip6_null_model = permutation_test(cmip6_delta_E, cmip6_delta_R, 6, 55)

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axvline(x=0, color="#4C4B4B", linestyle='--',)
plt.axhline(y=0, color='#4C4B4B', linestyle='--',)
plt.scatter(cmip5_delta_R, cmip5_delta_E,facecolors="#03727E", edgecolors='#000000',s=100,linewidths=1.2)
fit_line_confidence(cmip5_delta_R, cmip5_delta_E,color="#03727E",level=0.95)

plt.text(0.95,0.95,f'CMIP5',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.87,f'C = {cmip5_cg:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.79,f'$\Gamma$ = {cmip5_gamma:.2f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.71,rf'$p_{{\mathrm{{perm}}}}$ = {cmip5_p_perm:.4f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)

plt.ylim(-2.2,2.2)
plt.xlim(-1.7,1.7)
plt.xticks(np.arange(-1,1.1,1))
plt.yticks(np.arange(-2,2.1,1))
plt.xlabel('$\Delta R$ (K)', fontsize=18)
plt.ylabel('$\Delta E$ (K)', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG 2a.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axvline(x=0, color="#4C4B4B", linestyle='--',)
plt.axhline(y=0, color='#4C4B4B', linestyle='--',)
plt.scatter(cmip6_delta_R, cmip6_delta_E,facecolors="#BB6644", edgecolors='#000000',s=100,linewidths=1.2)
fit_line_confidence(cmip6_delta_R, cmip6_delta_E,color="#BB6644",level=0.95)

plt.text(0.95,0.30,f'CMIP6',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.22,f'C = {cmip6_cg:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.14,f'$\Gamma$ = {cmip6_gamma:.2f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(0.95,0.06,rf'$p_{{\mathrm{{perm}}}}$ = {cmip6_p_perm:.4f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)

plt.ylim(-2.2,2.2)
plt.xlim(-1.7,1.7)
plt.xticks(np.arange(-1,1.1,1))
plt.yticks(np.arange(-2,2.1,1))
plt.xlabel('$\Delta R$ (K)', fontsize=18)
plt.ylabel('$\Delta E$ (K)', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG 2b.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.subplots(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

kde = gaussian_kde(cmip5_null_model)
kde.set_bandwidth(bw_method=kde.factor *2)
x = np.linspace(-0.8,0.8,500)
pdf = kde(x)
plt.plot(x, pdf, color='#2C677D',linewidth=2)

kde = gaussian_kde(cmip6_null_model)
kde.set_bandwidth(bw_method=kde.factor *2)
x = np.linspace(-0.8,0.8,500)
pdf = kde(x)
plt.plot(x, pdf, color='#C4633E',linewidth=2)


plt.hist(cmip5_null_model,bins=30,density=True,histtype='step',linewidth=1.5,color='#2C677D',alpha = 0.3)
plt.hist(cmip5_null_model,bins=30,density=True,alpha=0.15,color='#2C677D')

plt.hist(cmip6_null_model,bins=30,density=True,histtype='step',linewidth=1.5,color='#C4633E',alpha = 0.3)
plt.hist(cmip6_null_model,bins=30,density=True,alpha=0.15,color='#C4633E')


plt.axvline(cmip5_cg,color='#2C677D',linewidth=2)
plt.axvline(cmip6_cg,color='#C4633E',linewidth=2)

plt.text(cmip5_cg+0.02,0.3,'CMIP5',rotation=90, color='#03727E',fontsize=18)
plt.text(cmip6_cg+0.02,0.3,'CMIP6',rotation=90,color='#BB6644',fontsize=18)

plt.ylabel('Probability density', fontsize=18)
plt.xlabel(r'$C$ under random pairing', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG 2c.svg', transparent=True,bbox_inches='tight')

In [ ]:
def build_A_B(params):
    lam = params["feedback_lambda"]
    b = params["direct_gain"]
    ct = params["heat_capacity_t"]
    cz = params["heat_capacity_z"]
    mu = params["hidden_damping"]
    kap = params["hidden_coupling"]
    q = params["hidden_forcing"]

    a, c, d = -lam/ct, kap/ct, -mu/cz
    A = np.stack((a, c, np.zeros_like(a), d), axis=-1).reshape(np.shape(a) + (2, 2))
    B = np.stack((b/ct, q/cz), axis=-1)
    return A, B

def matrix_expm(a, c, d, tau):
    ea, ed = np.exp(a*tau), np.exp(d*tau)
    gap = d-a
    off = c * ea * np.divide(np.expm1(gap*tau), gap, out=np.full_like(a, tau), where=gap != 0)
    return np.stack((ea, off, np.zeros_like(ea), ed), axis=-1).reshape(np.shape(a) + (2, 2))

def stationary_covariance(a, c, d, sigma_t, sigma_z, ct, cz):
    s22 = -(sigma_z/cz)**2 / (2*d)
    s12 = -c*s22 / (a+d)
    s11 = -((sigma_t/ct)**2 + 2*c*s12) / (2*a)
    return np.stack((s11, s12, s12, s22), axis=-1).reshape(np.shape(s11) + (2, 2))

def compute_psi(cov, prop_1):
    var_t = cov[..., 0, 0]
    lag1 = (prop_1 @ cov)[..., 0, 0] / var_t
    if not np.all((lag1 > 0) & (lag1 < 1)):
        raise ValueError("The stationary lag-one correlation must lie in (0, 1).")
    return np.sqrt(var_t) / np.sqrt(-np.log(lag1))

def compute_ecs_and_response(A, B, f2x, response_horizon):
    a, c, d = A[..., 0, 0], A[..., 0, 1], A[..., 1, 1]
    prop_tau = matrix_expm(a, c, d, tau=float(response_horizon))
    forcing = B[..., None]
    eq = -f2x * np.linalg.solve(A, forcing)
    fin = -f2x * np.linalg.solve(A, (np.eye(2) - prop_tau) @ forcing)
    return eq[..., 0, 0], fin[..., 0, 0]

def compute_model_diagnostics(params, f2x=3.7, response_horizon=55):
    A, B = build_A_B(params)
    a, c, d = A[..., 0, 0], A[..., 0, 1], A[..., 1, 1]
    cov = stationary_covariance(
        a, c, d, params["noise_t"], params["noise_z"],
        params["heat_capacity_t"], params["heat_capacity_z"],
    )
    psi = compute_psi(cov, matrix_expm(a, c, d, 1.0))
    ecs, response = compute_ecs_and_response(A, B, f2x, response_horizon)
    return {"psi": psi, "response": response, "ecs": ecs}

In [ ]:
def _zscore(values):
    return (values - values.mean()) / values.std(ddof=1)

def sample_projected_alignment_ensemble(
        n_models, seed, hidden_log_sd, projected_alignment, heat_capacity_log_sd,
        noise_log_sd=0.08, hidden_amplitude_center=0.20, f2x=3.7, response_horizon=55,
        fixed_hidden_mean=False):

    rng = np.random.default_rng(seed)

    feedback_lambda = np.exp(rng.normal(np.log(1.18), 0.22, n_models))
    direct_gain     = np.exp(rng.normal(np.log(0.88), 0.03, n_models))
    heat_capacity_t = np.exp(rng.normal(np.log(45.0), heat_capacity_log_sd, n_models))
    heat_capacity_z = np.exp(rng.normal(np.log(90.0), 0.22, n_models))
    hidden_damping  = np.exp(rng.normal(np.log(1.0), 0.14, n_models))
    hidden_coupling = np.exp(rng.normal(np.log(0.32), 0.18, n_models))
    noise_t = np.exp(rng.normal(np.log(0.28), noise_log_sd, n_models))
    noise_z = np.exp(rng.normal(np.log(0.075), noise_log_sd, n_models))

    a = -feedback_lambda / heat_capacity_t
    c = hidden_coupling / heat_capacity_t
    d = -hidden_damping / heat_capacity_z
    cov = stationary_covariance(a, c, d, noise_t, noise_z, heat_capacity_t, heat_capacity_z)
    provisional_psi = compute_psi(cov, matrix_expm(a, c, d, 1.0))
    psi_coordinate = _zscore(provisional_psi)

    indep_coord = rng.normal(0.0, 1.0, n_models)
    rho = projected_alignment
    s_perp = hidden_log_sd * np.sqrt(1.0 - rho**2)
    correlated = hidden_log_sd * rho * psi_coordinate
    hidden_amplitude = hidden_amplitude_center * np.exp(correlated + s_perp * indep_coord)
    if fixed_hidden_mean:
        hidden_amplitude /= np.exp(correlated).mean() * np.exp(s_perp**2 / 2)

    hidden_forcing = hidden_amplitude * feedback_lambda * hidden_damping / hidden_coupling
    params = {
        "feedback_lambda": feedback_lambda, "direct_gain": direct_gain,
        "heat_capacity_t": heat_capacity_t, "heat_capacity_z": heat_capacity_z,
        "hidden_damping": hidden_damping, "hidden_coupling": hidden_coupling,
        "hidden_forcing": hidden_forcing, "noise_t": noise_t, "noise_z": noise_z,
    }
    diag = compute_model_diagnostics(params, f2x, response_horizon)
    return pd.DataFrame({"hidden_amplitude": hidden_amplitude, **params, **diag})

In [ ]:
def fit_linear(x, y):
    X = np.column_stack([np.ones(len(x)), x])
    coef = np.linalg.lstsq(X, y, rcond=None)[0]
    return float(coef[0]), float(coef[1])

def add_self_closure(df):
    psi      = df["psi"].to_numpy(dtype=float)
    ecs      = df["ecs"].to_numpy(dtype=float)
    response = df["response"].to_numpy(dtype=float)

    intercept_e, slope_e = fit_linear(psi, ecs)
    intercept_r, slope_r = fit_linear(psi, response)

    df = df.copy()
    df["ecs_pred"]      = intercept_e + slope_e * psi
    df["response_pred"] = intercept_r + slope_r * psi
    df["delta_e"]       = ecs      - df["ecs_pred"]
    df["delta_r"]       = response - df["response_pred"]
    return df

def safe_corr(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan
    x, y = x[mask], y[mask]
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])

def fit_through_origin(x, y):
    denom = float(np.sum(x * x))
    if denom < 1e-12:
        return np.nan
    return float(np.sum(x * y) / denom)

def breaking_metrics(df):
    de = df["delta_e"].to_numpy(dtype=float)
    dr = df["delta_r"].to_numpy(dtype=float)
    C     = safe_corr(de, dr)
    Gamma = fit_through_origin(dr, de)
    return {"C": C, "Gamma": Gamma}

In [ ]:
def add_response_decomposition(df, f2x=3.7, response_horizon=55):

    df   = df.copy()
    tau  = float(response_horizon)

    B_arr = df["direct_gain"].to_numpy(float) / df["feedback_lambda"].to_numpy(float)
    H_arr = df["hidden_amplitude"].to_numpy(float)

    a_arr = -df["feedback_lambda"].to_numpy(float) / df["heat_capacity_t"].to_numpy(float)
    d_arr = -df["hidden_damping"].to_numpy(float)  / df["heat_capacity_z"].to_numpy(float)

    G_T_arr = 1.0 - np.exp(a_arr * tau) 

    ea = np.exp(a_arr * tau)
    gap = d_arr - a_arr
    exp_ratio = np.divide(np.expm1(gap * tau), gap, out=np.full_like(gap, tau), where=gap != 0)
    G_z_arr = G_T_arr + a_arr * ea * exp_ratio


    G_T_mean = float(np.mean(G_T_arr))
    G_z_mean = float(np.mean(G_z_arr))

    response_amp = f2x * (G_T_mean * B_arr + G_z_mean * H_arr)
    response_kin = f2x * (B_arr * (G_T_arr - G_T_mean)
                        + H_arr * (G_z_arr - G_z_mean))

    df["G_T"]          = G_T_arr
    df["G_z"]          = G_z_arr
    df["response_amp"] = response_amp
    df["response_kin"] = response_kin


    psi = df["psi"].to_numpy(float)
    for col in ["response_amp", "response_kin"]:
        ic, sl = fit_linear(psi, df[col].to_numpy(float))
        df[f"r_{col.split('_')[1]}"] = df[col].to_numpy(float) - (ic + sl * psi)

    r_amp = df["r_amp"].to_numpy(float)
    r_kin = df["r_kin"].to_numpy(float)

    var_amp = float(np.mean(r_amp ** 2))
    var_kin = float(np.mean(r_kin ** 2))
    Q       = var_amp / var_kin
    C_pred  = float(np.sqrt(Q / (1.0 + Q)))

    delta_e = df["delta_e"].to_numpy(float)
    delta_r = df["delta_r"].to_numpy(float)
    c_a = safe_corr(delta_e, r_amp)
    c_k = safe_corr(delta_e, r_kin)
    c_ak = safe_corr(r_amp, r_kin)
    exact_c = (c_a * np.sqrt(Q) + c_k) / np.sqrt(Q + 1 + 2 * c_ak * np.sqrt(Q))
    np.testing.assert_allclose(r_amp + r_kin, delta_r, rtol=0, atol=1e-12)
    np.testing.assert_allclose(exact_c, safe_corr(delta_e, delta_r), rtol=0, atol=1e-12)
    df.attrs.update(c_a=c_a, c_k=c_k, c_ak=c_ak, exact_c=exact_c)

    df["Q_amp_over_kin"] = Q
    df["C_pred"]         = C_pred
    return df, Q, C_pred

In [ ]:
def build_regime_grid_new(n_models=420, n_seed_avg=8, seed=20260617,
                          f2x=3.7, response_horizon=55):
    hidden_spreads  = np.round(np.linspace(0.05, 0.95, 31), 3)
    kinetic_spreads = np.round(np.linspace(0.12, 0.60, 31), 3)
    projected_alignment = 0.05

    rows = []
    for i_s, s in enumerate(hidden_spreads):
        for i_k, k in enumerate(kinetic_spreads):
            idx = len(kinetic_spreads) * i_s + i_k
            C_list, Q_list, Cpred_list, fixed_mean_list, terms = [], [], [], [], []
            for rep in range(n_seed_avg):
                params = dict(
                    n_models=n_models, seed=seed + idx * 100 + rep, hidden_log_sd=float(s),
                    projected_alignment=projected_alignment, heat_capacity_log_sd=float(k),
                    f2x=f2x, response_horizon=response_horizon,
                )
                df = add_self_closure(sample_projected_alignment_ensemble(**params))
                decomposed, Q, C_pred = add_response_decomposition(df, f2x, response_horizon)
                C_list.append(breaking_metrics(df)["C"])
                Q_list.append(Q)
                Cpred_list.append(C_pred)
                terms.append([decomposed.attrs[key] for key in ("c_a", "c_k", "c_ak")])

                fixed = add_self_closure(sample_projected_alignment_ensemble(**params, fixed_hidden_mean=True))
                fixed_mean_list.append(breaking_metrics(fixed)["C"])
            c_a, c_k, c_ak = np.mean(terms, axis=0)
            rows.append({
                "hidden_spread": float(s),
                "kinetic_spread": float(k),
                "unclosed_hidden_spread": float(s) * np.sqrt(1.0 - projected_alignment**2),
                "corr_deltae_deltar": float(np.mean(C_list)),
                "Q_amp_over_kin": float(np.mean(Q_list)),
                "C_simple_prediction": float(np.mean(Cpred_list)),
                "C_fixed_H_mean": float(np.mean(fixed_mean_list)),
                "c_a": c_a, "c_k": c_k, "c_ak": c_ak,
            })
    return pd.DataFrame(rows)

In [ ]:
result_df = build_regime_grid_new(seed=20260617 , f2x=3.7, response_horizon=55)

In [ ]:
representatives = pd.DataFrame([
    {"hidden_spread": 0.17, "kinetic_spread": 0.552, "seed": 20260617,
     "regime_id": "fdt_like", "regime_label": "scalar-closed"},
    {"hidden_spread": 0.83, "kinetic_spread": 0.184, "seed": 20261618,
     "regime_id": "hidden_response", "regime_label": "closure breaking"},
])
representatives["s_perp"] = representatives["hidden_spread"] * np.sqrt(1 - 0.05**2)

residual_rows = []
metric_rows   = []

for _, row in representatives.iterrows():
    df = sample_projected_alignment_ensemble(
        n_models=900,
        seed=int(row['seed']),
        hidden_log_sd=float(row["hidden_spread"]),
        projected_alignment=0.05,
        heat_capacity_log_sd=float(row["kinetic_spread"]),
        noise_log_sd=0.08,
        f2x=3.7,
        response_horizon=55,
    )

    df = add_self_closure(df)
    m  = breaking_metrics(df)

    residual_rows.append(pd.DataFrame({
        "regime_id":    str(row["regime_id"]),
        "regime_label": str(row["regime_label"]),
        "ecs" : df["ecs"].to_numpy(float),
        "psi" : df["psi"].to_numpy(float),
        "response" : df["response"].to_numpy(float),
        "delta_e":      df["delta_e"].to_numpy(float),
        "delta_r":      df["delta_r"].to_numpy(float),
    }))

    metric_rows.append({
        "regime_id":    str(row["regime_id"]),
        "regime_label": str(row["regime_label"]),
        "C":            m["C"],
        "Gamma":        m["Gamma"],
    })

residuals = pd.concat(residual_rows, ignore_index=True)
metrics   = pd.DataFrame(metric_rows)

In [ ]:
pivot = result_df.pivot(
    index="kinetic_spread",
    columns="unclosed_hidden_spread",
    values="corr_deltae_deltar",
)

In [ ]:
y = pivot.index.to_numpy()
x = pivot.columns.to_numpy()

fig,ax = plt.subplots(figsize=(5,4))
image = ax.imshow(
    pivot.to_numpy(),
    origin="lower",
    aspect="auto",
    extent=[
        x[0] - (x[1]-x[0])/2,
        x[-1] + (x[1]-x[0])/2,
        y[0] - (y[1]-y[0])/2,
        y[-1] + (y[1]-y[0])/2,
    ],
    cmap='GnBu',
    vmin = 0,
    vmax = 1,
    interpolation="nearest",
    alpha=1
)

ax.contour(
    pivot.columns.to_numpy(dtype=float),
    pivot.index.to_numpy(dtype=float),
    pivot.to_numpy(),
    levels=[0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
    colors="#FFFFFF",
    linewidths=1,
    linestyles="--",)

cbar = ax.figure.colorbar(image,ax=ax, fraction=0.046, pad=0.025,)
cbar.set_label(r"$C$", fontsize=18)
cbar.ax.tick_params(labelsize=18)
marker_annotations = (
    ("#03535C", r"$\Psi$-dominated", (0, -10), "center", "top"),
    ("#D36337", "Response-complementary", (8, 10), "right", "bottom"),
)
for row, (color, label, offset, ha, va) in zip(representatives.itertuples(), marker_annotations):
    plt.scatter(row.s_perp, row.kinetic_spread, facecolor="#FFFFFF", s=200, zorder=7)
    plt.scatter(row.s_perp, row.kinetic_spread, facecolor=color, s=100, zorder=7)
    ax.annotate(
        label,
        xy=(row.s_perp, row.kinetic_spread),
        xytext=offset,
        textcoords="offset points",
        ha=ha,
        va=va,
        color=color,
        fontsize=12,
        zorder=8,
    )

plt.xlabel(r'Unresolved hidden spread $s_\perp$', fontsize=18)
plt.ylabel(r'Heat-capacity log-spread $k$', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG 3a.svg', transparent=True,bbox_inches='tight')

In [ ]:
hidden_e = np.array(residuals.loc[(residuals['regime_id'] == 'hidden_response'),'ecs'])
hidden_psi = np.array(residuals.loc[(residuals['regime_id'] == 'hidden_response'),'psi'])
hidden_r = np.array(residuals.loc[(residuals['regime_id'] == 'hidden_response'),'response'])
hidden_e_psi_cor = np.corrcoef(hidden_e,hidden_psi)[0,1]
hidden_e_r_cor = np.corrcoef(hidden_e,hidden_r)[0,1]

fdt_like_e = np.array(residuals.loc[(residuals['regime_id'] == 'fdt_like'),'ecs'])
fdt_like_psi = np.array(residuals.loc[(residuals['regime_id'] == 'fdt_like'),'psi'])
fdt_like_r = np.array(residuals.loc[(residuals['regime_id'] == 'fdt_like'),'response'])
fdt_like_e_psi_cor = np.corrcoef(fdt_like_e,fdt_like_psi)[0,1]
fdt_like_e_r_cor = np.corrcoef(fdt_like_e,fdt_like_r)[0,1]

In [ ]:
hidden_delta_e = np.array(residuals.loc[(residuals['regime_id'] == 'hidden_response'),'delta_e'])
hidden_delta_r = np.array(residuals.loc[(residuals['regime_id'] == 'hidden_response'),'delta_r'])
fdt_like_delta_e = np.array(residuals.loc[(residuals['regime_id'] == 'fdt_like'),'delta_e'])
fdt_like_delta_r = np.array(residuals.loc[(residuals['regime_id'] == 'fdt_like'),'delta_r'])
fdt_like_C = metrics.loc[(metrics['regime_id'] == 'fdt_like'),"C"].values[0]
fdt_like_gamma = metrics.loc[(metrics['regime_id'] == 'fdt_like'),"Gamma"].values[0]
hidden_C = metrics.loc[(metrics['regime_id'] == 'hidden_response'),"C"].values[0]
hidden_gamma = metrics.loc[(metrics['regime_id'] == 'hidden_response'),"Gamma"].values[0]

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.scatter(fdt_like_delta_r,fdt_like_delta_e,color = "#03727E",alpha =0.2,s=10)
plt.scatter(hidden_delta_r,hidden_delta_e,color = "#BB6644",alpha =0.25,s=10)

fit_x_fdt = np.arange(np.min(fdt_like_delta_r),np.max(fdt_like_delta_r),0.01)
fit_x_hidden = np.arange(np.min(hidden_delta_r),np.max(hidden_delta_r),0.01)
                                
fit_hidden_slope, fit_hidden_intercept  = np.polyfit(hidden_delta_r, hidden_delta_e, deg=1)
fit_fdt_like_slope, fit_fdt_like_intercept  = np.polyfit(fdt_like_delta_r,fdt_like_delta_e, deg=1)
fit_hidden = fit_x_hidden*fit_hidden_slope+fit_hidden_intercept
fit_fdt_like = fit_x_fdt *fit_fdt_like_slope+fit_fdt_like_intercept

line_fdt, = plt.plot(fit_x_fdt, fit_fdt_like, color='#03727E', linewidth=1.8,
                     label=r'$\Psi$-dominated' + '\n' + rf'$C$ = {fdt_like_C:.3f}, $\Gamma$ = {fdt_like_gamma:.3f}')
line_hidden, = plt.plot(fit_x_hidden, fit_hidden, color='#BB6644', linewidth=1.8,
                        label='Response-complementary\n' + rf'$C$ = {hidden_C:.3f}, $\Gamma$ = {hidden_gamma:.3f}')

plt.axvline(0,color='black',linewidth=1.2, alpha = 0.3,linestyle='--')
plt.axhline(0,color='black',linewidth=1.2, alpha = 0.3,linestyle='--')

all_delta_e = np.r_[fdt_like_delta_e, hidden_delta_e]
all_delta_r = np.r_[fdt_like_delta_r, hidden_delta_r]
plt.ylim(min(-2, all_delta_e.min()-0.2), max(5, all_delta_e.max()+0.3))
plt.xlim(min(-1.8, all_delta_r.min()-0.1), max(2.1, all_delta_r.max()+0.1))
plt.xticks(np.arange(-1.5,2.1,1))
plt.yticks(np.arange(-2, max(5, all_delta_e.max()+0.3), 2))
plt.xlabel(r'Finite-response residual $\Delta R$ (K)', fontsize=18)
plt.ylabel(r'Sensitivity residual $\Delta E$ (K)', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.legend(handles=[line_fdt, line_hidden], loc='upper left', frameon=False, fontsize=12)
plt.savefig(subgraph_dir / 'FIG 3b.svg', transparent=True,bbox_inches='tight')

In [ ]:
cmip6_E_corr = np.zeros((23)) 
for i in range (23):
    cmip6_E_corr[i]= (fit_E_psi_cmip6_slope * cmip6_psi[i] + fit_E_psi_cmip6_intercept)+cmip6_gamma*cmip6_delta_R[i]

s6 = np.sqrt(np.sum((cmip6_ecs - cmip6_E_corr)**2) / (len(cmip6_ecs) - 3))

In [ ]:
HadCRUT5_psi_mean = np.mean(HadCRUT5_psi)
HadCRUT5_cmip6_fit_E = (fit_E_psi_cmip6_slope * HadCRUT5_psi_mean + fit_E_psi_cmip6_intercept)
HadCRUT5_cmip6_fit_R = (fit_R_psi_cmip6_slope * HadCRUT5_psi_mean + fit_R_psi_cmip6_intercept)
HadCRUT5_delta_R = HadCRUT5_Rco2-HadCRUT5_cmip6_fit_R
HadCRUT5_E_corr = HadCRUT5_cmip6_fit_E+cmip6_gamma*HadCRUT5_delta_R

In [ ]:
cmip6_gamma

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.axvline(x = HadCRUT5_E_corr,color='#000000', linestyle='--',)
plt.scatter(HadCRUT5_E_corr,HadCRUT5_E_corr,facecolor = "#000000",edgecolors="#FFFFFF",label = 'HadCRUT5',marker ='D',s=150,linewidths=2,zorder=10)
plt.scatter(cmip6_E_corr, cmip6_ecs,facecolors="#BB6644", edgecolors='#000000',s=100,linewidths=1.2)
fit_x = np.linspace(cmip6_E_corr.min(), cmip6_E_corr.max(), 120)
plt.fill_between(fit_x, fit_x-1.96*s6, fit_x+1.96*s6, color="#BB6644", alpha=0.11, linewidth=0)
plt.plot(fit_x, fit_x, color="#BB6644", linewidth=1.15)
r, p = stats.pearsonr(cmip6_E_corr, cmip6_ecs)
plt.text(1,0.13,f'In-sample r = {r:.3f}',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)
plt.text(1,0.04,f'HadCRUT5: {HadCRUT5_E_corr:.2f} K',transform=plt.gca().transAxes,ha='right',va='bottom',fontsize=18)

plt.ylim(1.5,6)
plt.xlim(1.8,6)
plt.xlabel('Corrected ECS estimate (K)', fontsize=18)
plt.ylabel('Model ECS (K)', fontsize=18)
plt.yticks(np.arange(2,6.1,1))
plt.xticks(np.arange(2,6.1,1))
plt.tick_params(axis='both', which='major', labelsize=18)
plt.savefig(subgraph_dir / 'FIG 4a.svg', transparent=True,bbox_inches='tight')

In [ ]:
ARX_DRAWS = 12_000
ARX_DRAW_SEED = 20260528
MONTE_CARLO_DRAWS = 50_000
MONTE_CARLO_SEED = 20261112

def Rco2_samples(tas, forcing, horizon=75, n_samples=ARX_DRAWS, seed=ARX_DRAW_SEED):
    best = select_arx(tas, forcing)
    p = best["order"]
    coef = np.r_[best["intercept"], best["ar_coef"], best["forcing_coef"]]
    design = _arx_design(tas, forcing, order=p)
    residual = tas[p:] - design @ coef
    n, k = design.shape
    cov = (residual @ residual) / (n-k) * np.linalg.pinv(design.T @ design)
    cov = 0.5 * (cov + cov.T)

    rng = np.random.default_rng(seed)
    accepted = []
    remaining, rejected = n_samples, 0
    while remaining:
        draws = rng.multivariate_normal(coef, cov, size=remaining, method="svd")
        ar = draws[:, 1:1+p]
        stable = np.abs(ar[:, 0]) < 1 if p == 1 else np.apply_along_axis(_spectral_radius, 1, ar) < 1
        accepted.append(draws[stable])
        rejected += int(np.count_nonzero(~stable))
        remaining -= int(np.count_nonzero(stable))
        if rejected > 100*n_samples:
            raise RuntimeError("Could not obtain enough stable ARX draws.")
    draws = np.concatenate(accepted)
    ar = draws[:, 1:1+p]
    co2_step = F2XCO2 * draws[:, 1+p]
    states = np.zeros_like(ar)
    curves = np.zeros((n_samples, horizon+1))
    for year in range(1, horizon+1):
        curves[:, year] = np.sum(ar * states, axis=1) + co2_step
        states[:, 1:] = states[:, :-1]
        states[:, 0] = curves[:, year]
    center = np.r_[0, F2XCO2 * step_response_curve(best["ar_coef"], best["forcing_coef"][0], horizon=horizon)]
    return center, curves, rejected

In [ ]:
hadcrut_center_curve, hadcrut_response_draws, hadcrut_rejected = Rco2_samples(
    HadCRUT5_1900_2014, forcing_1900_2014, horizon=75
)
r_center = hadcrut_center_curve[55]
r_samples = hadcrut_response_draws[:, 55]

In [ ]:
np.std(r_samples)

In [ ]:
def batch_ols(x, y):
    mx, my = x.mean(axis=1), y.mean(axis=1)
    dx, dy = x-mx[:, None], y-my[:, None]
    slope = np.sum(dx*dy, axis=1) / np.sum(dx**2, axis=1)
    intercept = my - slope*mx
    return slope, intercept, dy - slope[:, None]*dx

def bootstrap_calibration(psi, response, ecs, indices):
    psi_b, r_b, e_b = psi[indices], response[indices], ecs[indices]
    e_slope, e_intercept, de = batch_ols(psi_b, e_b)
    r_slope, r_intercept, dr = batch_ols(psi_b, r_b)
    er_slope, er_intercept, _ = batch_ols(r_b, e_b)
    gamma = np.sum(de*dr, axis=1) / np.sum(dr**2, axis=1)
    return {
        "E0": (e_slope, e_intercept), "R0": (r_slope, r_intercept),
        "Gamma": gamma, "E_from_R": (er_slope, er_intercept),
    }

def sample_observed(psi_windows, response_pool, rng):
    psi_draw = rng.choice(psi_windows[np.isfinite(psi_windows)], MONTE_CARLO_DRAWS)
    r_draw = rng.choice(response_pool, MONTE_CARLO_DRAWS)
    return psi_draw, r_draw

def corrected_samples(psi_draw, r_draw, calibration):
    fit_ob_E = np.polyval(calibration["E0"], psi_draw)
    fit_ob_R = np.polyval(calibration["R0"], psi_draw)
    return fit_ob_E + calibration["Gamma"]*(r_draw - fit_ob_R)

bootstrap_rng = np.random.default_rng(MONTE_CARLO_SEED)
cmip6_bootstrap_indices = bootstrap_rng.integers(
    len(cmip6_ecs), size=(MONTE_CARLO_DRAWS, len(cmip6_ecs))
)
cmip6_bootstrap = bootstrap_calibration(cmip6_psi, cmip6_Rco2, cmip6_ecs, cmip6_bootstrap_indices)

psi_draw, r_draw = sample_observed(
    HadCRUT5_psi, r_samples, np.random.default_rng([MONTE_CARLO_SEED, 2014, 55, 1])
)
x_samples = corrected_samples(psi_draw, r_draw, cmip6_bootstrap)

In [ ]:
residual_rng = np.random.default_rng([MONTE_CARLO_SEED, 2014, 55, 2])
ECS_samples = x_samples + residual_rng.normal(0.0, s6, size=x_samples.size)

In [ ]:
median_ecs = np.median(ECS_samples)

low66 = np.percentile(
    ECS_samples,
    17
)

high66 = np.percentile(
    ECS_samples,
    83
)


In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
bins = [ 1,1.5, 2,2.5, 3,3.5, 4,4.5, 5,5.5, 6]
plt.hist(cmip6_ecs,density=True,bins=bins,color = "#BB6644",alpha = 0.8,label='CMIP6')
plt.axvline(x = HadCRUT5_E_corr,color='#000000', linestyle='--',)
x = np.sort(np.r_[np.linspace(1,6,500), low66, high66])

pdf = np.array([
    np.mean(np.exp(-0.5*((value-x_samples)/s6)**2)) / (np.sqrt(2*np.pi)*s6)
    for value in x
])

mask = (x >= low66) & (x <= high66)

plt.plot(x, pdf, color='black',label='Corrected density',linewidth = 2)

plt.fill_between(
    x[mask],
    0,
    pdf[mask],
    color='black',
    alpha=0.15,
    label = 'Corrected 66% interval'
)
plt.hlines(y=0.47, xmin=2.5, xmax=4, colors='red', linewidth=3, linestyles="-", label='AR6 likely range')
plt.vlines(x=2.5, ymin=0.45, ymax=0.49, colors='red', linewidth=3,linestyles="-")
plt.vlines(x=4, ymin=0.45, ymax=0.49, colors='red', linewidth=3,linestyles="-")
plt.scatter(3,0.47,color='red',s = 80,zorder = 7,edgecolors='black',label='AR6 best estimate')
plt.xlabel("ECS (K)", fontsize=18)
plt.ylabel("Probability density", fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)
handles, labels = plt.gca().get_legend_handles_labels()
legend_handles = dict(zip(labels, handles))
legend_order = ['AR6 likely range', 'AR6 best estimate', 'CMIP6', 'Corrected 66% interval', 'Corrected density']
plt.legend([legend_handles[label] for label in legend_order], legend_order,
           fontsize=10, frameon=0, loc='upper right', bbox_to_anchor=(1.05,1))
plt.savefig(subgraph_dir / 'FIG 4b.svg', transparent=True,bbox_inches='tight')


In [ ]:
cmip5_years = np.arange(1900, 1900 + cmip5_tas.shape[1])
cmip6_years = np.arange(1900, 1900 + cmip6_tas.shape[1])
hadcrut_years = np.arange(1900, 1900 + HadCRUT5_1900_2014.size)

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

for i in range (cmip5_tas.shape[0]):
    plt.plot(cmip5_years,cmip5_tas[i,:],color = "#03727E",alpha = 0.11)
for i in range (cmip6_tas.shape[0]):
    plt.plot(cmip6_years,cmip6_tas[i,:],color = "#BB6644",alpha = 0.08)

plt.plot(cmip5_years,np.mean(cmip5_tas,axis=0),color = "#03727E",linewidth = 2.5,label = 'CMIP5')
plt.plot(cmip6_years,np.mean(cmip6_tas,axis=0),color = "#BB6644",linewidth = 2.5,label = 'CMIP6')
plt.plot(hadcrut_years,HadCRUT5_1900_2014,color = "#000000",linewidth = 2.5,label = 'HadCRUT5')

plt.ylabel('Temperature anomaly (K)', fontsize=18)
plt.xlabel('Year', fontsize=18)
plt.xticks(
    [1900,1950,2000],
    ["1900","1950","2000"]
)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.legend(loc='upper left', frameon=False,fontsize=18)

plt.savefig(subgraph_dir / 'FIG S1a.svg', transparent=True,bbox_inches='tight')

In [ ]:
def cmip6_calibration_check(psi, response, ecs, observed_response, case):
    E0, R0, de, dr = residual_pair(psi, ecs, response)
    c = np.corrcoef(de, dr)[0, 1]
    gamma = (dr @ de) / (dr @ dr)
    p_perm, _ = permutation_test(de, dr, 6, 55, case=case)
    point = np.polyval(E0, HadCRUT5_psi_mean) + gamma*(observed_response - np.polyval(R0, HadCRUT5_psi_mean))
    return {"C6": c, "p_perm": p_perm, "HadCRUT5 ECS (K)": point}

In [ ]:
cmip6_catalog = model_ecs.loc[cmip6_model_name_dirs]
meehl_only = cmip6_catalog['ecs_source'].eq('Meehl Table 2').to_numpy()
cmip6_zelinka = cmip6_catalog['zelinka_v2_2'].to_numpy()
assert np.count_nonzero(meehl_only) == 22 and np.isfinite(cmip6_zelinka).all()

ecs_source_sensitivity = {
    "Meehl 22": cmip6_calibration_check(
        cmip6_psi[meehl_only], cmip6_Rco2[meehl_only], cmip6_ecs[meehl_only], HadCRUT5_Rco2, case=1
    ),
    "Zelinka v2.2": cmip6_calibration_check(cmip6_psi, cmip6_Rco2, cmip6_zelinka, HadCRUT5_Rco2, case=2),
}

In [ ]:
display(pd.DataFrame.from_dict(ecs_source_sensitivity, orient='index'))

In [ ]:
cmip5_psi_low, cmip5_psi_high = np.percentile(cmip5_psi, [16, 84])
cmip6_psi_low, cmip6_psi_high = np.percentile(cmip6_psi, [16, 84])

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.axhspan(cmip5_psi_low, cmip5_psi_high, color="#03727E", alpha=0.08)
plt.axhspan(cmip6_psi_low, cmip6_psi_high, color="#BB6644", alpha=0.15)
plt.axhline(np.mean(cmip5_psi), color="#03727E", linestyle='--' ,linewidth = 2.5)
plt.axhline(np.mean(cmip6_psi), color="#BB6644", linestyle='--',linewidth = 2.5)
midpoint_years = np.arange(1927, 1927 + len(HadCRUT5_psi))
plt.axhline(np.mean(HadCRUT5_psi), color="#000000", linestyle='--', linewidth=1.5)
plt.plot(midpoint_years,HadCRUT5_psi,color = "#000000",linewidth = 2.5,)
plt.scatter(np.mean(midpoint_years),np.mean(HadCRUT5_psi),facecolor = "#000000",edgecolors="#FFFFFF",label = 'HadCRUT5 mean',marker ='D',s=150,linewidths=2,zorder=10)
plt.ylabel(r'Window $\Psi$ (K)', fontsize=18)
plt.xlabel('Window midpoint', fontsize=18)
plt.xticks(
    [1940,1960,1980],
    ["1940","1960","1980"]
)

plt.tick_params(axis='both', which='major', labelsize=18)
plt.legend(loc='lower left', frameon=False,fontsize=18)

plt.savefig(subgraph_dir / 'FIG S1b.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

bins = np.histogram_bin_edges(np.r_[cmip5_psi, cmip6_psi], bins=12)
pad = 0.08 * (bins[-1] - bins[0])
psi_grid = np.linspace(max(0.0, bins[0] - pad), bins[-1] + pad, 500)
for values, label, color in [
    (cmip5_psi, 'CMIP5', '#2C677D'),
    (cmip6_psi, 'CMIP6', '#C4633E'),
]:
    kde = gaussian_kde(values)
    kde.set_bandwidth(bw_method=2 * kde.factor)
    plt.hist(values, bins=bins, density=True, color=color, alpha=0.15)
    plt.hist(values, bins=bins, density=True, histtype='step', linewidth=1.5, color=color, alpha=0.3)
    plt.plot(psi_grid, kde(psi_grid), color=color, linewidth=2, label=label)

plt.axvline(np.mean(HadCRUT5_psi), color='#000000', linestyle='--', linewidth=1.5, label='HadCRUT5')
plt.xlabel(r'Record-mean $\Psi$ (K)', fontsize=18)
plt.ylabel('Probability density', fontsize=18)
plt.xticks([0.1, 0.2, 0.3, 0.4])
plt.tick_params(axis='both', which='major', labelsize=18)
plt.legend(loc='upper right', frameon=False, fontsize=14)
plt.savefig(subgraph_dir / 'FIG S1c.svg', transparent=True,bbox_inches='tight')

In [ ]:
cmip5_ARX_order = np.array([select_arx(tas, forcing_1900_2005)["order"] for tas in cmip5_tas])
cmip6_ARX_order = np.array([select_arx(tas, forcing_1900_2014)["order"] for tas in cmip6_tas])
HadCRUT5_ARX_order = select_arx(HadCRUT5_1900_2014, forcing_1900_2014)["order"]

cmip5_plateau_change = np.max(np.abs((cmip5_Rco2 - cmip5_Rco2_step[:, -1]) / cmip5_Rco2_step[:, -1]))
cmip6_plateau_change = np.max(np.abs((cmip6_Rco2 - cmip6_Rco2_step[:, -1]) / cmip6_Rco2_step[:, -1]))
display(pd.DataFrame([
    {"Ensemble": "CMIP5", **{f"p={p}": np.count_nonzero(cmip5_ARX_order == p) for p in (1, 2, 3)},
     "Max relative change, 55 to 150 yr": cmip5_plateau_change},
    {"Ensemble": "CMIP6", **{f"p={p}": np.count_nonzero(cmip6_ARX_order == p) for p in (1, 2, 3)},
     "Max relative change, 55 to 150 yr": cmip6_plateau_change},
]))

In [ ]:
cmip6_Rco2_common = np.array([
    Rco2_step(tas, forcing_1900_2014, horizon=55, target_start=3)[-1] for tas in cmip6_tas
])
hadcrut_Rco2_common = Rco2_step(HadCRUT5_1900_2014, forcing_1900_2014, horizon=55, target_start=3)[-1]
common_start = cmip6_calibration_check(
    cmip6_psi, cmip6_Rco2_common, cmip6_ecs, hadcrut_Rco2_common, case=3
)
display(pd.DataFrame([common_start], index=["Common ARX target years"]))

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

years_after_step = np.arange(151)
cmip5_step_plot = np.column_stack([np.zeros(len(cmip5_Rco2_step)), cmip5_Rco2_step])
cmip6_step_plot = np.column_stack([np.zeros(len(cmip6_Rco2_step)), cmip6_Rco2_step])
cmip5_step_low, cmip5_step_high = np.percentile(cmip5_step_plot, [16, 84], axis=0)
cmip6_step_low, cmip6_step_high = np.percentile(cmip6_step_plot, [16, 84], axis=0)

plt.plot(years_after_step, np.mean(cmip5_step_plot, axis=0), color='#03727E', label=f'CMIP5 (n = {len(cmip5_step_plot)})', linewidth=2)
plt.plot(years_after_step, np.mean(cmip6_step_plot, axis=0), color='#BB6644', label=f'CMIP6 (n = {len(cmip6_step_plot)})', linewidth=2)
plt.fill_between(years_after_step, cmip5_step_low, cmip5_step_high, color='#03727E', alpha=0.2)
plt.fill_between(years_after_step, cmip6_step_low, cmip6_step_high, color='#BB6644', alpha=0.25)
plt.axvline(55, color="#4C4B4B", linestyle='--', linewidth=1.2)

plt.ylabel('Estimated temperature\nresponse (K)', fontsize=18)
plt.xlabel('Years after the reference forcing step', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.legend(loc='lower right', frameon=False,fontsize=14)

plt.ylim(0,5)
plt.xlim(0,150)
plt.xticks([0, 25, 55, 100, 150])
plt.text(57, 0.32, '55 years', fontsize=14)

plt.savefig(subgraph_dir / 'FIG S2a.svg', transparent=True,bbox_inches='tight')
plt.savefig(subgraph_dir / 'FIG S2a.png', dpi=180, bbox_inches='tight')


In [ ]:
from itertools import combinations

def leave_k_out(psi, response, ecs, k):
    correlations = []
    for omitted in combinations(range(len(ecs)), k):
        keep = np.ones(len(ecs), dtype=bool)
        keep[list(omitted)] = False
        _, _, de, dr = residual_pair(psi[keep], ecs[keep], response[keep])
        correlations.append(np.corrcoef(de, dr)[0, 1])
    return np.array(correlations)

cmip5_leave_1, cmip5_leave_2, cmip5_leave_3 = [
    leave_k_out(cmip5_psi, cmip5_Rco2, cmip5_ecs, k) for k in (1, 2, 3)
]

In [ ]:
cmip6_leave_1, cmip6_leave_2, cmip6_leave_3 = [
    leave_k_out(cmip6_psi, cmip6_Rco2, cmip6_ecs, k) for k in (1, 2, 3)
]

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

vp = plt.violinplot(
    [cmip5_leave_1, cmip6_leave_1],
    showextrema=False,
    widths=0.5,
    showmeans=False,
    positions=[1, 1.5],
)

colors = ["#03727E", "#BB6644"]

for body, color in zip(vp['bodies'], colors):
    body.set_facecolor(color)
    body.set_edgecolor('none')
    body.set_alpha(0.3)


bp = plt.boxplot(
    [cmip5_leave_1, cmip6_leave_1],
    positions=[1, 1.5],
    showfliers=False,
    whis=(0, 100),
    widths=0.2,       
    patch_artist=True,
    boxprops=dict(facecolor="#FFFFFF", alpha=1), 
    medianprops=dict(color='black', linewidth=1), 
    whiskerprops=dict(color='black'),          
    capprops=dict(color='black'))

plt.hlines(y=cmip5_cg, xmin=0.7, xmax=1.3, colors='#03727E', linewidth=2,linestyles="--")
plt.hlines(y=cmip6_cg, xmin=1.2, xmax=1.8, colors='#BB6644', linewidth=2,linestyles="--")
plt.axhline(0,color = "#636363",linewidth=1.5,linestyle="--")

plt.xticks([1, 1.5],['CMIP5', 'CMIP6'])
plt.ylim(-0.55,0.82)
plt.yticks(np.arange(-0.5,0.76,0.25))
plt.ylabel(r'$C$ after refitting', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG S3a.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

vp = plt.violinplot(
    [cmip5_leave_2, cmip6_leave_2],
    showextrema=False,
    widths=0.5,
    showmeans=False,
    positions=[1, 1.5],
)

colors = ["#03727E", "#BB6644"]

for body, color in zip(vp['bodies'], colors):
    body.set_facecolor(color)
    body.set_edgecolor('none')
    body.set_alpha(0.3)


bp = plt.boxplot(
    [cmip5_leave_2, cmip6_leave_2],
    positions=[1, 1.5],
    showfliers=False,
    whis=(0, 100),
    widths=0.2,       
    patch_artist=True,
    boxprops=dict(facecolor="#FFFFFF", alpha=1), 
    medianprops=dict(color='black', linewidth=1), 
    whiskerprops=dict(color='black'),          
    capprops=dict(color='black'))

plt.hlines(y=cmip5_cg, xmin=0.7, xmax=1.3, colors='#03727E', linewidth=2,linestyles="--")
plt.hlines(y=cmip6_cg, xmin=1.2, xmax=1.8, colors='#BB6644', linewidth=2,linestyles="--")
plt.axhline(0,color = "#636363",linewidth=1.5,linestyle="--")

plt.xticks([1, 1.5],['CMIP5', 'CMIP6'])
plt.ylim(-0.55,0.82)
plt.yticks(np.arange(-0.5,0.76,0.25))
plt.ylabel(r'$C$ after refitting', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG S3b.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

vp = plt.violinplot(
    [cmip5_leave_3, cmip6_leave_3],
    showextrema=False,
    widths=0.5,
    showmeans=False,
    positions=[1, 1.5],
)

colors = ["#03727E", "#BB6644"]

for body, color in zip(vp['bodies'], colors):
    body.set_facecolor(color)
    body.set_edgecolor('none')
    body.set_alpha(0.3)


bp = plt.boxplot(
    [cmip5_leave_3, cmip6_leave_3],
    positions=[1, 1.5],
    showfliers=False,
    whis=(0, 100),
    widths=0.2,       
    patch_artist=True,
    boxprops=dict(facecolor="#FFFFFF", alpha=1), 
    medianprops=dict(color='black', linewidth=1), 
    whiskerprops=dict(color='black'),          
    capprops=dict(color='black'))

plt.hlines(y=cmip5_cg, xmin=0.7, xmax=1.3, colors='#03727E', linewidth=2,linestyles="--")
plt.hlines(y=cmip6_cg, xmin=1.2, xmax=1.8, colors='#BB6644', linewidth=2,linestyles="--")
plt.axhline(0,color = "#636363",linewidth=1.5,linestyle="--")

plt.xticks([1, 1.5],['CMIP5', 'CMIP6'])
plt.ylim(-0.55,0.82)
plt.yticks(np.arange(-0.5,0.76,0.25))
plt.ylabel(r'$C$ after refitting', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG S3c.svg', transparent=True,bbox_inches='tight')

In [ ]:
window_size_lsit = [45, 50, 55, 60, 65, 70, 75]

def window_complementarity(window_psi, window_response, ecs, group, p_55):
    coefficients, p_values = [], []
    for width, psi, response in zip(window_size_lsit, window_psi, window_response):
        _, _, de, dr = residual_pair(psi, ecs, response)
        coefficients.append(np.corrcoef(de, dr)[0, 1])
        if width == 55:
            p_values.append(p_55)
        else:
            p_value, _ = permutation_test(de, dr, group, width)
            p_values.append(p_value)
    return np.array(coefficients), np.array(p_values)

In [ ]:
cmip5_window_values = {
    width: np.stack([rolling_psi(tas, width) for tas in cmip5_tas])
    for width in window_size_lsit
}
window_cmip5_psi = np.stack([
    record_mean_psi(cmip5_window_values[width]) for width in window_size_lsit
])

In [ ]:
window_cmip5_Rco2 = cmip5_Rco2_step[:, np.array(window_size_lsit) - 1].T

In [ ]:
cmip5_window_cg, cmip5_window_P_perm = window_complementarity(
    window_cmip5_psi, window_cmip5_Rco2, cmip5_ecs, 5, cmip5_p_perm
)

In [ ]:
cmip6_window_values = {
    width: np.stack([rolling_psi(tas, width) for tas in cmip6_tas])
    for width in window_size_lsit
}
window_cmip6_psi = np.stack([
    record_mean_psi(cmip6_window_values[width]) for width in window_size_lsit
])

In [ ]:
window_cmip6_Rco2 = cmip6_Rco2_step[:, np.array(window_size_lsit) - 1].T

In [ ]:
cmip6_window_cg, cmip6_window_P_perm = window_complementarity(
    window_cmip6_psi, window_cmip6_Rco2, cmip6_ecs, 6, cmip6_p_perm
)

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

x = window_size_lsit
plt.plot(x,cmip5_window_cg,color='#03727E',linewidth=2,label='CMIP5')
plt.plot(x,cmip6_window_cg,color='#BB6644',linewidth=2,label='CMIP6')
plt.scatter(x,cmip5_window_cg,facecolor='#ffffff',edgecolor = "black",s=120,zorder = 7)
plt.scatter(x,cmip6_window_cg,facecolor="#BB6644",edgecolor = "black",s=120,zorder = 7)

plt.axhline(0,color = "#636363",linewidth=2,linestyle="--")

plt.ylabel(r'Complementarity coefficient $C$', fontsize=18)
plt.xlabel('Window / response horizon (yr)', fontsize=18)
plt.xticks(x)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.legend(fontsize=14,frameon=False)
plt.savefig(subgraph_dir / 'FIG S4a.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

x = window_size_lsit
plt.plot(x,cmip5_window_P_perm,color='#03727E',linewidth=2)
plt.plot(x,cmip6_window_P_perm,color='#BB6644',linewidth=2)
plt.scatter(x,cmip5_window_P_perm,facecolor='#03727E',edgecolor = "black",s=120,zorder = 7)
plt.scatter(x,cmip6_window_P_perm,facecolor="#BB6644",edgecolor = "black",s=120,zorder = 7)

plt.yscale('log')
plt.ylim(0.0001, 1.5)

plt.axhline(0.05,color = "#636363",linewidth=1.5,linestyle="--")
plt.text(45,0.0005,r"below line: $p_{\rm perm}<0.05$",fontsize=14)

legend_elements = [
Line2D([0], [0], color='#03727E',markerfacecolor = "#03727E",markeredgecolor = 'black', lw=2, marker='o', markersize=8, label='CMIP5'),
Line2D([0], [0], color='#BB6644',markerfacecolor = "#BB6644",markeredgecolor = 'black', lw=2, marker='o', markersize=8, label='CMIP6'),
]
plt.ylabel(r'Two-sided $p_{\rm perm}$', fontsize=18)
plt.xlabel('Window / response horizon (yr)', fontsize=18)
plt.xticks(x)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.legend(handles=legend_elements,fontsize=14,frameon=False,loc = 'lower right')

plt.savefig(subgraph_dir / 'FIG S4b.svg', transparent=True,bbox_inches='tight')

In [ ]:
from scipy.optimize import curve_fit

def fit_linear_ref(psi, R):
    X = np.column_stack([np.ones(len(psi)), psi])
    coef = np.linalg.lstsq(X, R, rcond=None)[0]
    return coef

def predict_linear(psi, coef):
    return coef[0] + coef[1] * psi

def onebox_ref(psi, B, A):
    return B * psi * (1.0 - np.exp(-A / psi))

def fit_onebox_ref(psi, R):
    B0 = float(np.mean(R / psi))
    A0 = float(np.mean(psi))
    popt, _ = curve_fit(onebox_ref, psi, R, p0=[B0, A0],
                        bounds=([0, 1e-6], [100, 100]),
                        maxfev=5000)
    return popt

def predict_onebox(psi, popt):
    return onebox_ref(psi, *popt)

def fit_quad_ref(psi, R):
    X = np.column_stack([np.ones(len(psi)), psi, psi**2])
    coef = np.linalg.lstsq(X, R, rcond=None)[0]
    return coef

def predict_quad(psi, coef):
    return coef[0] + coef[1]*psi + coef[2]*psi**2

psi = cmip5_psi
R   = cmip5_Rco2
ecs = cmip5_ecs

X_psi   = np.column_stack([np.ones(len(psi)), psi])
E0_coef = np.linalg.lstsq(X_psi, ecs, rcond=None)[0]
delta_E = ecs - X_psi @ E0_coef

def metrics_from_R0(R, R0_pred, delta_E):
    delta_R = R - R0_pred
    rmse_R  = float(np.sqrt(np.mean((R - R0_pred)**2)))
    C       = float(np.corrcoef(delta_E, delta_R)[0, 1])
    dr, de = delta_R - delta_R.mean(), delta_E - delta_E.mean()
    gamma = float((dr @ de) / (dr @ dr))
    return rmse_R, C, gamma

cmip5_lin_coef  = fit_linear_ref(psi, R)
cmip5_r1, cmip5_c1, cmip5_g1 = metrics_from_R0(R, predict_linear(psi, cmip5_lin_coef), delta_E)

cmip5_ob_popt   = fit_onebox_ref(psi, R)
cmip5_r2, cmip5_c2, cmip5_g2 = metrics_from_R0(R, predict_onebox(psi, cmip5_ob_popt), delta_E)

cmip5_q_coef    = fit_quad_ref(psi, R)
cmip5_r3, cmip5_c3, cmip5_g3 = metrics_from_R0(R, predict_quad(psi, cmip5_q_coef), delta_E)

In [ ]:
psi = cmip6_psi
R   = cmip6_Rco2
ecs = cmip6_ecs

X_psi   = np.column_stack([np.ones(len(psi)), psi])
E0_coef = np.linalg.lstsq(X_psi, ecs, rcond=None)[0]
delta_E = ecs - X_psi @ E0_coef

cmip6_lin_coef  = fit_linear_ref(psi, R)
cmip6_r1, cmip6_c1, cmip6_g1 = metrics_from_R0(R, predict_linear(psi, cmip6_lin_coef), delta_E)

cmip6_ob_popt   = fit_onebox_ref(psi, R)
cmip6_r2, cmip6_c2, cmip6_g2 = metrics_from_R0(R, predict_onebox(psi, cmip6_ob_popt), delta_E)

cmip6_q_coef    = fit_quad_ref(psi, R)
cmip6_r3, cmip6_c3, cmip6_g3 = metrics_from_R0(R, predict_quad(psi, cmip6_q_coef), delta_E)

In [ ]:
psi_range5 = np.linspace(cmip5_psi.min()*0.95, cmip5_psi.max()*1.05, 200)

curves5 = {
    "linear":    predict_linear(psi_range5, cmip5_lin_coef),
    "one-box":   predict_onebox(psi_range5, cmip5_ob_popt),
    "quadratic": predict_quad(psi_range5,   cmip5_q_coef),
}

styles = {
    "linear":    {"color": "#222222", "ls": "-",  "lw": 2, "label": "linear"},
    "one-box":   {"color": "#4C78A8", "ls": "--", "lw": 2, "label": "one-box"},
    "quadratic": {"color": "#C46243", "ls": ":",  "lw": 2, "label": "quadratic"},

}

fig = plt.figure(figsize=(6,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.scatter(cmip5_psi, cmip5_Rco2, s=100, color="#03727E",
           edgecolors="#202020", linewidths=1, zorder=3)
plt.title("CMIP5", fontsize=18)
for name, kw in styles.items():
    plt.plot(psi_range5, curves5[name], **kw)
plt.xlabel(r"$\Psi$ (K)", fontsize=18)
plt.ylabel(r"$R$ (K)", fontsize=18)
plt.legend(fontsize=14, frameon=False)
plt.tick_params(labelsize=18)

plt.savefig(subgraph_dir / 'FIG S5a.svg', transparent=True,bbox_inches='tight')

In [ ]:
psi_range6 = np.linspace(cmip6_psi.min()*0.95, cmip6_psi.max()*1.05, 200)

curves6 = {
    "linear":    predict_linear(psi_range6, cmip6_lin_coef),
    "one-box":   predict_onebox(psi_range6, cmip6_ob_popt),
    "quadratic": predict_quad(psi_range6,   cmip6_q_coef),
}

styles = {
    "linear":    {"color": "#222222", "ls": "-",  "lw": 2, "label": "linear"},
    "one-box":   {"color": "#4C78A8", "ls": "--", "lw": 2, "label": "one-box"},
    "quadratic": {"color": "#C46243", "ls": ":",  "lw": 2, "label": "quadratic"},

}

fig = plt.figure(figsize=(6,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.scatter(cmip6_psi, cmip6_Rco2, s=100, color="#BB6644",
           edgecolors="#202020", linewidths=1, zorder=3)
plt.title("CMIP6", fontsize=18)
for name, kw in styles.items():
    plt.plot(psi_range6, curves6[name], **kw)
plt.xlabel(r"$\Psi$ (K)", fontsize=18)
plt.ylabel(r"$R$ (K)", fontsize=18)
plt.legend(fontsize=14, frameon=False)
plt.tick_params(labelsize=18)
plt.savefig(subgraph_dir / 'FIG S5b.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

labels  = ["linear", "one-box", "quadratic"]
x       = np.arange(len(labels))
width   = 0.35

gamma5 = [cmip5_g1,cmip5_g2,cmip5_g3]
gamma6 = [cmip6_g1,cmip6_g2,cmip6_g3]

plt.bar(x - width/2, gamma5, width, color="#03727E", label="CMIP5",
       edgecolor="black", linewidth=0.5)
plt.bar(x + width/2, gamma6, width, color="#BB6644", label="CMIP6",
       edgecolor="black", linewidth=0.5)

plt.xticks(x,labels)
plt.ylabel(r"$\Gamma$", fontsize=18)
plt.legend(fontsize=14, frameon=False)
plt.tick_params(labelsize=18)
plt.savefig(subgraph_dir / 'FIG S5c.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

labels  = ["linear", "one-box", "quadratic"]
x       = np.arange(len(labels))
width   = 0.35

cg5 = [cmip5_c1,cmip5_c2,cmip5_c3]
cg6 = [cmip6_c1,cmip6_c2,cmip6_c3]

plt.bar(x - width/2, cg5, width, color="#03727E", label="CMIP5",
       edgecolor="black", linewidth=0.5)
plt.bar(x + width/2, cg6, width, color="#BB6644", label="CMIP6",
       edgecolor="black", linewidth=0.5)

plt.ylim(0,0.8)
plt.xticks(x,labels)
plt.ylabel(r"$C$", fontsize=18)
plt.legend(fontsize=14, frameon=False,loc = 'upper center')
plt.tick_params(labelsize=18)

plt.savefig(subgraph_dir / 'FIG S5d.svg', transparent=True,bbox_inches='tight')

In [ ]:
scalar_closed_ecs = np.array(residuals[residuals['regime_label'] == 'scalar-closed']['ecs'])
scalar_closed_psi = np.array(residuals[residuals['regime_label'] == 'scalar-closed']['psi'])
scalar_closed_R = np.array(residuals[residuals['regime_label'] == 'scalar-closed']['response'])
scalar_closed_delta_R = np.array(residuals[residuals['regime_label'] == 'scalar-closed']['delta_r'])
scalar_closed_delta_E = np.array(residuals[residuals['regime_label'] == 'scalar-closed']['delta_e'])
scalar_closed_cg = np.corrcoef(scalar_closed_delta_E,scalar_closed_delta_R)[0,1]

closure_breaking_ecs = np.array(residuals[residuals['regime_label'] == 'closure breaking']['ecs'])
closure_breaking_psi = np.array(residuals[residuals['regime_label'] == 'closure breaking']['psi'])
closure_breaking_R = np.array(residuals[residuals['regime_label'] == 'closure breaking']['response'])
closure_breaking_delta_R = np.array(residuals[residuals['regime_label'] == 'closure breaking']['delta_r'])
closure_breaking_delta_E = np.array(residuals[residuals['regime_label'] == 'closure breaking']['delta_e'])
closure_breaking_cg = np.corrcoef(closure_breaking_delta_E,closure_breaking_delta_R)[0,1]

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)


plt.scatter(scalar_closed_ecs, scalar_closed_psi,facecolors="#03727E", edgecolors='#000000',s=30)
fit_line(scalar_closed_ecs, scalar_closed_psi,color="#03727E",level=0.95)
r, _ = stats.pearsonr(scalar_closed_ecs, scalar_closed_psi)
plt.text(0.05, 0.95, r'$\Psi$-dominated' + f'\nr = {r:.3f}',
         transform=plt.gca().transAxes, ha='left', va='top', fontsize=16)

plt.ylim(0.04,0.4)
plt.xlim(1.5, max(6.5, scalar_closed_ecs.max()+0.35))
plt.xticks([2, 3, 4, 5, 6])
plt.ylabel(r'$\Psi$ (K)', fontsize=18)
plt.xlabel(r'Equilibrium warming $E$ (K)', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG S6a.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)


plt.scatter(closure_breaking_ecs, closure_breaking_psi,facecolors="#BB6644", edgecolors='#000000',s=30)
fit_line(closure_breaking_ecs, closure_breaking_psi,color="#BB6644",level=0.95)
r, _ = stats.pearsonr(closure_breaking_ecs, closure_breaking_psi)
plt.text(0.05, 0.95, 'Response-complementary' + f'\nr = {r:.3f}',
         transform=plt.gca().transAxes, ha='left', va='top', fontsize=16)

plt.ylim(0.04,0.4)
plt.xlim(1.5, max(8, scalar_closed_ecs.max()+0.5, closure_breaking_ecs.max()+0.5))
plt.xticks(np.arange(2, max(8.1, closure_breaking_ecs.max()+0.5), 2))
plt.ylabel(r'$\Psi$ (K)', fontsize=18)
plt.xlabel(r'Equilibrium warming $E$ (K)', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG S6b.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)


plt.scatter(scalar_closed_ecs, scalar_closed_R,facecolors="#03727E", edgecolors='#000000',s=30)
fit_line(scalar_closed_ecs, scalar_closed_R,color="#03727E",level=0.95)
r, _ = stats.pearsonr(scalar_closed_ecs, scalar_closed_R)
plt.text(0.05, 0.95, r'$\Psi$-dominated' + f'\nr = {r:.3f}',
         transform=plt.gca().transAxes, ha='left', va='top', fontsize=16)

plt.ylabel(r'$R$ (K)', fontsize=18)
plt.xlabel(r'Equilibrium warming $E$ (K)', fontsize=18)
plt.xlim(1.5, max(6.5, scalar_closed_ecs.max()+0.35))
plt.xticks([2, 3, 4, 5, 6])
plt.yticks(np.arange(1,4.1,1))
plt.tick_params(axis='both', which='major', labelsize=18)
plt.savefig(subgraph_dir / 'FIG S6c.svg', transparent=True,bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)


plt.scatter(closure_breaking_ecs, closure_breaking_R,facecolors="#BB6644", edgecolors='#000000',s=30)
fit_line(closure_breaking_ecs, closure_breaking_R,color="#BB6644",level=0.95)
r, _ = stats.pearsonr(closure_breaking_ecs, closure_breaking_R)
plt.text(0.05, 0.95, 'Response-complementary' + f'\nr = {r:.3f}',
         transform=plt.gca().transAxes, ha='left', va='top', fontsize=16)

plt.ylabel(r'$R$ (K)', fontsize=18)
plt.xlabel(r'Equilibrium warming $E$ (K)', fontsize=18)
plt.xlim(1.5, max(8, scalar_closed_ecs.max()+0.5, closure_breaking_ecs.max()+0.5))
plt.xticks(np.arange(2, max(8.1, closure_breaking_ecs.max()+0.5), 2))
plt.yticks(np.arange(2,4.1,1))
plt.tick_params(axis='both', which='major', labelsize=18)
plt.savefig(subgraph_dir / 'FIG S6d.svg', transparent=True,bbox_inches='tight')

In [ ]:
x_pred = result_df["C_simple_prediction"].to_numpy()
y_obs  = result_df["corr_deltae_deltar"].to_numpy()
color  = result_df["unclosed_hidden_spread"].to_numpy()

fig, ax = plt.subplots(figsize=(5, 4))

sc = ax.scatter(x_pred, y_obs, c=color, cmap="magma", s=15, alpha=0.7,vmin = 0,vmax = 1,edgecolors='none')
cbar = ax.figure.colorbar(sc,ax=ax, fraction=0.046, pad=0.025,)
cbar.set_label(r"$s_\perp$", fontsize=16)
cbar.ax.tick_params(labelsize=18)
ax.plot([0.1, 0.9], [0.1, 0.9], "k--", linewidth=1.2)
r, p = stats.pearsonr(x_pred[np.isfinite(x_pred) & np.isfinite(y_obs)],
                       y_obs [np.isfinite(x_pred) & np.isfinite(y_obs)])
error = x_pred - y_obs
toy_rmse = np.sqrt(np.mean(error**2))
toy_approximation_r = r
ax.text(0.05, 0.82, f"r = {r:.3f}\nRMSE = {toy_rmse:.3f}", transform=ax.transAxes, fontsize=18)
ax.set_xlabel(r"Approximation $\sqrt{Q/(1+Q)}$", fontsize=18)
ax.set_ylabel(r"Calculated $C$", fontsize=18)
plt.ylim(0,1)
plt.xlim(0.1,0.9)
plt.yticks(np.arange(0,1.1,0.2))
plt.xticks(np.arange(0.2,1,0.2))
plt.tick_params(axis='both', which='major', labelsize=18)
plt.savefig(subgraph_dir / 'FIG S7.svg', transparent=True,bbox_inches='tight')

In [ ]:
window = np.array(window_size_lsit)
window_point = np.empty(len(window))
ecs_samples_array = np.empty((len(window), MONTE_CARLO_DRAWS))

for i, width in enumerate(window):
    if width == 55:
        window_point[i] = HadCRUT5_E_corr
        ecs_samples_array[i] = ECS_samples
        continue

    psi_model = window_cmip6_psi[i]
    response_model = window_cmip6_Rco2[i]
    E0, R0, de, dr = residual_pair(psi_model, cmip6_ecs, response_model)
    gamma = (dr @ de) / (dr @ dr)
    sigma = np.linalg.norm(de - gamma*dr) / np.sqrt(len(cmip6_ecs) - 3)

    psi_windows = rolling_psi(HadCRUT5_1900_2014, width)
    psi_point = record_mean_psi(psi_windows)
    r_point = hadcrut_center_curve[width]
    window_point[i] = np.polyval(E0, psi_point) + gamma*(r_point - np.polyval(R0, psi_point))

    rng = np.random.default_rng([MONTE_CARLO_SEED, 2014, int(width), 1])
    calibration = bootstrap_calibration(psi_model, response_model, cmip6_ecs, cmip6_bootstrap_indices)
    psi_w, r_w = sample_observed(psi_windows, hadcrut_response_draws[:, width], rng)
    centers = corrected_samples(psi_w, r_w, calibration)
    ecs_samples_array[i] = centers + rng.normal(0, sigma, MONTE_CARLO_DRAWS)

In [ ]:
window_low, window_high = np.percentile(ecs_samples_array, [17, 83], axis=1)
window_yerr = np.vstack([window_point-window_low, window_high-window_point])

In [ ]:
fig = plt.figure(figsize=(5,4))

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

window = np.array([45, 50, 55, 60, 65, 70, 75])
plt.errorbar(
    window,
    window_point,
    yerr=window_yerr,
    fmt='o-',
    color='#BB6644',
    ecolor='#BB6644',
    capsize=5,
    lw=1.5,
    ms=9
)

plt.errorbar(
    window[2],
    window_point[2],
    yerr=[[window_yerr[0,2]],[window_yerr[1,2]]],
    fmt='o-',
    color="#000000",
    ecolor="#000000",
    capsize=5,
    lw=1.5,
    ms=9
)

plt.axhspan(2.5, 4, color="#DC4010", alpha=0.08)
plt.axhline(3,color="#DC4010", linestyle = '--')

plt.ylim(1.8,4.1)

plt.xlabel("Window / response horizon (yr)", fontsize=18)
plt.ylabel("Conditional ECS estimate (K)", fontsize=18)
plt.xticks(window)
plt.yticks(np.arange(2,4.1,0.5))
plt.tick_params(axis='both', which='major', labelsize=18)

plt.savefig(subgraph_dir / 'FIG S8a.svg', transparent=True,bbox_inches='tight')


In [ ]:
fit_E_R_cmip6 = np.polyfit(cmip6_Rco2, cmip6_ecs, 1)
psi_only = np.polyval(cmip6_bootstrap["E0"], psi_draw)
R_only = np.polyval(cmip6_bootstrap["E_from_R"], r_draw)

sigma_psi = np.linalg.norm(cmip6_delta_E) / np.sqrt(len(cmip6_ecs) - 2)
sigma_R = np.linalg.norm(cmip6_ecs - np.polyval(fit_E_R_cmip6, cmip6_Rco2)) / np.sqrt(len(cmip6_ecs) - 2)
single_rng = np.random.default_rng([MONTE_CARLO_SEED, 2014, 55, 3])
ECS_psi_only = psi_only + single_rng.normal(0.0, sigma_psi, MONTE_CARLO_DRAWS)
ECS_R_only = R_only + single_rng.normal(0.0, sigma_R, MONTE_CARLO_DRAWS)

In [ ]:
predictor_points = np.array([
    HadCRUT5_cmip6_fit_E,
    np.polyval(fit_E_R_cmip6, HadCRUT5_Rco2),
    HadCRUT5_E_corr,
])
predictor_low, predictor_high = np.percentile([ECS_psi_only, ECS_R_only, ECS_samples], [17, 83], axis=1)
predictor_yerr = np.vstack([predictor_points-predictor_low, predictor_high-predictor_points])

In [ ]:
fig = plt.figure(figsize=(5,4))

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

loc=[1,2,3]  
model = [r"$\Psi$ only", "R only", "Combined"]
plt.errorbar(
    loc,
    predictor_points,
    yerr=predictor_yerr,
    fmt='o',
    color='#BB6644',
    ecolor='#BB6644',
    capsize=5,
    lw=1.5,
    ms=9
)

plt.axhspan(2.5, 4, color="#DC4010", alpha=0.08)
plt.axhline(3,color="#DC4010", linestyle = '--')

plt.ylim(1.9,4.4)
  
plt.ylabel("Conditional ECS estimate (K)", fontsize=18)
plt.xlabel("CMIP6 calibration", fontsize=18)
plt.xticks(loc,model)
plt.yticks(np.arange(2,4.1,0.5))
plt.tick_params(axis='both', which='major', labelsize=18)
plt.savefig(subgraph_dir / 'FIG S8b.svg', transparent=True,bbox_inches='tight')

In [ ]:
def ECS_estimate(tas, forcing, end_year):
    windows = rolling_psi(tas, 55)
    psi_point = record_mean_psi(windows)
    center_curve, r_pool, _ = Rco2_samples(tas, forcing, horizon=55, seed=[ARX_DRAW_SEED, end_year])
    r_point = center_curve[55]
    point = np.polyval(cmip6_E0, psi_point) + cmip6_gamma*(r_point - np.polyval(cmip6_R0, psi_point))

    rng = np.random.default_rng([MONTE_CARLO_SEED, end_year, 55, 1])
    psi_end, r_end = sample_observed(windows, r_pool[:, 55], rng)
    centers = corrected_samples(psi_end, r_end, cmip6_bootstrap)
    draws = centers + rng.normal(0.0, s6, MONTE_CARLO_DRAWS)
    return point, draws

In [ ]:
forcing_1900_2019 = forcing_1900_to(2019)

In [ ]:
ECS_point_2005, ECS_estimate_2005 = ECS_estimate(HadCRUT5_1900_2014[:-9], forcing_1900_2005, 2005)
ECS_point_2014, ECS_estimate_2014 = HadCRUT5_E_corr, ECS_samples
ECS_point_2019, ECS_estimate_2019 = ECS_estimate(HadCRUT5_1900_2019, forcing_1900_2019, 2019)

In [ ]:
end_year_points = np.array([ECS_point_2005, ECS_point_2014, ECS_point_2019])
end_year_low, end_year_high = np.percentile(
    [ECS_estimate_2005, ECS_estimate_2014, ECS_estimate_2019], [17, 83], axis=1
)
end_year_yerr = np.vstack([end_year_points-end_year_low, end_year_high-end_year_points])

In [ ]:
fig = plt.figure(figsize=(5,4))

plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

loc=[1,2,3]  
model = ["2005", "2014", "2019"]
plt.errorbar(
    loc,
    end_year_points,
    yerr=end_year_yerr,
    fmt='o-',
    color='#BB6644',
    ecolor='#BB6644',
    capsize=5,
    lw=1.5,
    ms=9
)

plt.axhspan(2.5, 4, color="#DC4010", alpha=0.08)
plt.axhline(3,color="#DC4010", linestyle = '--')


plt.ylim(1.9,4.5)
  
plt.ylabel("Conditional ECS estimate (K)", fontsize=18)
plt.xlabel("HadCRUT5 end year (start: 1900)", fontsize=18)
plt.xticks(loc,model,rotation=15)
plt.yticks(np.arange(2,4.1,0.5))
plt.tick_params(axis='both', which='major', labelsize=18)
plt.savefig(subgraph_dir / 'FIG S8c.svg', transparent=True,bbox_inches='tight')

In [ ]:
checks = []

def compare(label, value, expected, atol=0.0006):
    np.testing.assert_allclose(value, expected, rtol=0, atol=atol, err_msg=label)
    checks.append(label)

def compare_p(label, value, expected):
    tolerance = 4 * np.sqrt(2 * expected * (1-expected) / PERMUTATIONS)
    compare(label, value, expected, atol=tolerance)

compare("55 年窗口数", [cmip5_psi_windows.shape[1], cmip6_psi_windows.shape[1], len(HadCRUT5_psi)],
        [52, 61, 61], atol=0)
compare("EC-Earth3-Veg-LR ECS", model_ecs.loc["EC-Earth3-Veg-LR", "ecs"], 4.23, atol=0)
compare("稳定 ARX 阶数", [np.bincount(cmip5_ARX_order)[1:], np.bincount(cmip6_ARX_order)[1:]],
        [[14, 1, 3], [12, 5, 6]], atol=0)
compare("HadCRUT5 ARX 阶数和拒绝抽样数", [HadCRUT5_ARX_order, hadcrut_rejected], [1, 0], atol=0)
assert max(cmip5_plateau_change, cmip6_plateau_change) < 4e-7
checks.append("55–150 年平台差异小于 4×10⁻⁷")
compare("Figure 1 边际相关",
        [np.corrcoef(a, b)[0, 1] for a,b in [(cmip5_psi, cmip5_ecs), (cmip6_psi, cmip6_ecs),
         (cmip5_Rco2, cmip5_ecs), (cmip6_Rco2, cmip6_ecs)]], [0.625, 0.400, 0.416, 0.661])
compare("主结果 C₅、C₆ 和 Γ₆", [cmip5_cg, cmip6_cg, cmip6_gamma], [0.152, 0.594, 0.901])
compare_p("CMIP5 置换 p", cmip5_p_perm, 0.54539)
compare_p("CMIP6 置换 p", cmip6_p_perm, 0.00364)

for label, arrays, counts, bounds in [
    ("CMIP5", (cmip5_leave_1, cmip5_leave_2, cmip5_leave_3), [18, 153, 816],
     [[0.018, 0.370], [-0.124, 0.486], [-0.240, 0.586]]),
    ("CMIP6", (cmip6_leave_1, cmip6_leave_2, cmip6_leave_3), [23, 253, 1771],
     [[0.529, 0.654], [0.483, 0.707], [0.444, 0.756]]),
]:
    compare(f"{label} 留 1–3 个模式组合数", [len(a) for a in arrays], counts, atol=0)
    compare(f"{label} 留 1–3 个模式 C 范围", [[a.min(), a.max()] for a in arrays], bounds)

compare("45–75 年 C 的范围",
        [[cmip5_window_cg.min(), cmip5_window_cg.max()], [cmip6_window_cg.min(), cmip6_window_cg.max()]],
        [[0.043, 0.202], [0.573, 0.607]])
compare("55 年窗口复用主结果",
        [cmip5_window_cg[2], cmip6_window_cg[2], cmip5_window_P_perm[2], cmip6_window_P_perm[2]],
        [cmip5_cg, cmip6_cg, cmip5_p_perm, cmip6_p_perm], atol=1e-12)
for name, windows, index, valid, total in [
    ("bcc-csm1-1, 45", cmip5_window_values[45], cmip5_model_name_dirs.index("bcc-csm1-1"), 61, 62),
    ("FGOALS-f3-L, 45", cmip6_window_values[45], cmip6_model_name_dirs.index("FGOALS-f3-L"), 52, 71),
    ("FGOALS-f3-L, 50", cmip6_window_values[50], cmip6_model_name_dirs.index("FGOALS-f3-L"), 56, 66),
]:
    compare(f"{name} 有效/总窗口数", [np.isfinite(windows[index]).sum(), windows.shape[1]],
            [valid, total], atol=0)
compare("表 S2 的 RMSE、C、Γ",
        [[cmip5_r1, cmip5_c1, cmip5_g1], [cmip5_r2, cmip5_c2, cmip5_g2], [cmip5_r3, cmip5_c3, cmip5_g3],
         [cmip6_r1, cmip6_c1, cmip6_g1], [cmip6_r2, cmip6_c2, cmip6_g2], [cmip6_r3, cmip6_c3, cmip6_g3]],
        [[0.874, 0.152, 0.098], [0.784, 0.121, 0.087], [0.769, 0.064, 0.047],
         [0.660, 0.594, 0.901], [0.692, 0.571, 0.827], [0.593, 0.646, 1.092]])
for name, outcome, expected in [
    ("Meehl 22", ecs_source_sensitivity["Meehl 22"], [0.605, 2.966, 0.00354]),
    ("Zelinka v2.2", ecs_source_sensitivity["Zelinka v2.2"], [0.589, 3.000, 0.00382]),
    ("共同 ARX 目标年份", common_start, [0.578, 3.008, 0.00470]),
]:
    compare(name, [outcome["C6"], outcome["HadCRUT5 ECS (K)"]], expected[:2])
    compare_p(name + " 置换 p", outcome["p_perm"], expected[2])

compare("双模态网格与两组示例",
        [len(result_df), fdt_like_C, fdt_like_gamma, hidden_C, hidden_gamma],
        [961, 0.106, 0.049, 0.674, 1.997])
compare("双模态边际相关", [fdt_like_e_psi_cor, fdt_like_e_r_cor, hidden_e_psi_cor, hidden_e_r_cor],
        [0.923, 0.385, 0.575, 0.781])
compare("双模态网格最大 C 与固定均值敏感性",
        [result_df["corr_deltae_deltar"].max(), result_df["C_fixed_H_mean"].max()], [0.875, 0.808])
compare("S7 近似式 r、偏差、RMSE 和最大误差",
        [toy_approximation_r, error.mean(), toy_rmse, np.abs(error).max()], [0.953, 0.107, 0.118, 0.215])

compare("HadCRUT5 Ψ、R、E₀、R₀、ECS",
        [HadCRUT5_psi_mean, HadCRUT5_Rco2, HadCRUT5_cmip6_fit_E, HadCRUT5_cmip6_fit_R, HadCRUT5_E_corr],
        [0.145204, 2.565804, 3.164518, 2.750657, 2.997903], atol=0.0002)
compare("HadCRUT5 条件散度及抽样次数", [s6, len(r_samples), len(ECS_samples)], [0.864737, 12000, 50000])
compare("HadCRUT5 66% 区间", [low66, high66], [2.106324, 3.911300], atol=0.025)
compare("S8a 点估计范围", [window_point.min(), window_point.max()], [2.942, 3.054])
compare("S8b 单预测量与联合点估计", predictor_points, [3.165, 3.089, 2.998])
compare("S8c 末年敏感性点估计", end_year_points, [3.347, 2.998, 3.120])

display(pd.DataFrame([
    {"样本": "CMIP5", "C": cmip5_cg, "Γ": cmip5_gamma, "p_perm": cmip5_p_perm},
    {"样本": "CMIP6", "C": cmip6_cg, "Γ": cmip6_gamma, "p_perm": cmip6_p_perm},
]))
print(f"HadCRUT5 ECS = {HadCRUT5_E_corr:.6f} K，66% 区间 [{low66:.4f}, {high66:.4f}] K")
print(f"通过 {len(checks)} 项与正文和补充材料的数值核对。")

In [ ]:
from scipy.optimize import least_squares

def Neural_ODE(tas, forcing, rng=None, return_fit=False):
    tas = np.asarray(tas, dtype=float)
    forcing = np.asarray(forcing, dtype=float)
    if tas.size != forcing.shape[0]:
        raise ValueError("Temperature and forcing must have matching rows.")
    if rng is None:
        rng = np.random.default_rng(20260522)
    temp_scale = max(float(np.std(tas, ddof=0)), 0.1)
    forcing_scales = np.maximum(np.std(forcing, axis=0, ddof=0), 0.1)
    scale = np.r_[temp_scale, forcing_scales]
    Xn = np.column_stack([tas[:-1], forcing[1:]]) / scale
    dT = np.diff(tas)
    input_dim, hidden_units = Xn.shape[1], 2

    def unpack(theta):
        offset = 1 + input_dim
        w1 = theta[offset:offset + input_dim * hidden_units].reshape(input_dim, hidden_units)
        offset += input_dim * hidden_units
        b1 = theta[offset:offset + hidden_units]
        w2 = theta[offset + hidden_units:offset + 2 * hidden_units]
        return theta[0], theta[1:1 + input_dim], w1, b1, w2

    def predict_dT(theta, design):
        c, a, w1, b1, w2 = unpack(theta)
        return c + design @ a + np.tanh(design @ w1 + b1) @ w2

    linear_design = np.column_stack([np.ones(len(dT)), Xn])
    coefficients, *_ = np.linalg.lstsq(linear_design, dT, rcond=None)
    w1 = rng.normal(0.0, 0.05, size=(input_dim, hidden_units))
    b1 = np.zeros(hidden_units)
    w2 = rng.normal(0.0, 0.01, size=hidden_units)
    theta0 = np.r_[coefficients, w1.ravel(), b1, w2]
    lower = np.full(theta0.size, -5.0)
    upper = np.full(theta0.size, 5.0)
    lower[0], upper[0] = -1.0, 1.0

    def residual(theta):
        return np.r_[predict_dT(theta, Xn) - dT, np.sqrt(0.1) * theta[1:]]

    res = least_squares(residual, theta0, bounds=(lower, upper),
                        max_nfev=500, xtol=1e-8, ftol=1e-8, gtol=1e-8)
    if not res.success:
        raise RuntimeError(f"Neural ODE optimization failed: {res.message}")
    theta = res.x

    def integrate(co2_forcing, n_year=150):
        T = 0.0
        trajectory = np.zeros(n_year)
        for year in range(n_year):
            z = np.array([[T, co2_forcing, 0.0, 0.0]]) / scale
            T += predict_dT(theta, z)[0]
            if not np.isfinite(T) or abs(T) > 50.0:
                trajectory[year:] = np.nan
                break
            trajectory[year] = T
        return trajectory

    control = integrate(0.0)
    step = integrate(3.93)
    response = step - control
    if not np.all(np.isfinite(response)):
        raise RuntimeError("Neural step/control trajectory exceeded the numerical guard.")
    _, _, w1_fit, b1_fit, w2_fit = unpack(theta)
    nonlinear = np.tanh(Xn @ w1_fit + b1_fit) @ w2_fit
    affine = np.column_stack([np.ones(len(Xn)), Xn])
    nonaffine = nonlinear - affine @ np.linalg.lstsq(affine, nonlinear, rcond=None)[0]
    diagnostics = {
        "success": bool(res.success), "nfev": int(res.nfev), "cost": float(res.cost),
        "parameters": theta.copy(), "scale": scale.copy(),
        "max_abs_nonlinear_output": float(np.max(np.abs(nonlinear))),
        "nonaffine_rms": float(np.sqrt(np.mean(nonaffine ** 2))),
    }
    return (response, diagnostics) if return_fit else response


In [ ]:
neural_rng = np.random.default_rng(20260522)
cmip5_neural_step = np.zeros((18, 150))
cmip5_neural_fits = []
for i in range(18):
    cmip5_neural_step[i], fit = Neural_ODE(
        cmip5_tas[i], forcing_1900_2005, rng=neural_rng, return_fit=True)
    cmip5_neural_fits.append(fit)

cmip6_neural_step = np.zeros((23, 150))
cmip6_neural_fits = []
for i in range(23):
    cmip6_neural_step[i], fit = Neural_ODE(
        cmip6_tas[i], forcing_1900_2014, rng=neural_rng, return_fit=True)
    cmip6_neural_fits.append(fit)

HadCRUT5_neural_step, HadCRUT5_neural_fit = Neural_ODE(
    HadCRUT5_1900_2014, forcing_1900_2014, rng=neural_rng, return_fit=True)

cmip5_low66, cmip5_high66 = np.percentile(cmip5_neural_step, [17, 83], axis=0)
cmip6_low66, cmip6_high66 = np.percentile(cmip6_neural_step, [17, 83], axis=0)
print("Reference settings: hidden_units=2, L2=0.1, max_nfev=500, seed=20260522")
display(pd.DataFrame([
    {"ensemble": ens, "converged": sum(f["success"] for f in fits), "models": len(fits),
     "nonaffine_RMS_above_1e-6": sum(f["nonaffine_rms"] > 1e-6 for f in fits),
     "max_abs_nonlinear_output_K_per_year": max(f["max_abs_nonlinear_output"] for f in fits)}
    for ens, fits in [("CMIP5", cmip5_neural_fits), ("CMIP6", cmip6_neural_fits)]
]))


In [ ]:
fig = plt.figure(figsize=(6,4))
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

r1, p1 = stats.pearsonr(cmip5_Rco2_step[:,54],cmip5_neural_step[:,54])
r2, p2 = stats.pearsonr(cmip6_Rco2_step[:,54],cmip6_neural_step[:,54])

plt.text(1.5,4.5,f"CMIP5: r={r1:.3f}",fontsize=14,color ="#03727E")
plt.text(1.5,4.1,f"CMIP6: r={r2:.3f}",fontsize=14,color ="#BB6644")

x_fit = np.concatenate((cmip5_Rco2_step[:,54],cmip6_Rco2_step[:,54]))
y_fit = np.concatenate((cmip5_neural_step[:,54],cmip6_neural_step[:,54]))
slope, intercept = np.polyfit(x_fit, y_fit, deg=1)
fit_x = np.arange(np.min(x_fit),np.max(x_fit),0.01)
fit_y = fit_x*slope+intercept

plt.scatter(cmip5_Rco2_step[:,54],cmip5_neural_step[:,54],facecolor='#03727E',edgecolors='#000000',s=80,label = "CMIP5")
plt.scatter(cmip6_Rco2_step[:,54],cmip6_neural_step[:,54],facecolor='#BB6644',edgecolors='#000000',s=80,label = "CMIP6")

plt.plot(fit_x,fit_y,color="#4C4B4B",linestyle="--",linewidth = 2)

plt.ylabel(r'Neural ODE $R(55)$ (K)', fontsize=18)
plt.xlabel(r'ARX $R(55)$ (K)', fontsize=18)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.legend(loc='lower right', frameon=False,fontsize=14)

plt.savefig(subgraph_dir / 'FIG Neural ODE comparison.svg', transparent=True,bbox_inches='tight')
plt.savefig(subgraph_dir / 'FIG Neural ODE comparison.png', dpi=180, bbox_inches='tight')

def _neural_residual_correlation(psi, ecs, response):
    psi = np.asarray(psi)
    if psi.ndim == 2:
        psi = psi.mean(axis=1)
    design = np.column_stack([np.ones(len(psi)), psi])
    e = np.asarray(ecs) - design @ np.linalg.lstsq(design, ecs, rcond=None)[0]
    r = response - design @ np.linalg.lstsq(design, response, rcond=None)[0]
    return float(np.corrcoef(e, r)[0, 1])

neural_comparison_results = pd.DataFrame([
    {"ensemble": ens, "ARX_vs_Neural_r": float(r), "ARX_vs_Neural_p": float(p),
     "ARX_Cg": _neural_residual_correlation(psi, ecs, arx[:, 54]),
     "Neural_Cg": _neural_residual_correlation(psi, ecs, neural[:, 54])}
    for ens, psi, ecs, arx, neural, r, p in [
        ("CMIP5", cmip5_psi, cmip5_ecs, cmip5_Rco2_step, cmip5_neural_step, r1, p1),
        ("CMIP6", cmip6_psi, cmip6_ecs, cmip6_Rco2_step, cmip6_neural_step, r2, p2),
    ]
])
display(neural_comparison_results)
print(f"HadCRUT5: Neural R55={HadCRUT5_neural_step[54]:.6f} K; "
      f"ARX R55={HadCRUT5_Rco2_step[54]:.6f} K")
